# Analisi delle partite LEC 2025

Il presente notebook documenta l'analisi esplorativa e modellistica condotta sul dataset delle partite della *League of Legends EMEA Championship* (LEC) relative alla stagione competitiva 2025. L'analisi è organizzata in quattro sezioni: una prima parte dedicata al **contesto competitivo generale**, una seconda alle **performance dei team**, una terza alle **performance individuali dei giocatori** e una quarta dedicata alla **modellazione predittiva dell'esito delle partite** mediante tecniche di *machine learning*.

Ogni domanda di ricerca è introdotta da un titolo dedicato, seguita dal codice di preparazione dei dati, dal grafico associato e da una sezione intitolata **Osservazione grafico**, in cui vengono discussi i principali risultati emersi dall'analisi.

## Configurazione dell'ambiente di analisi

Vengono importate le librerie necessarie per la manipolazione dei dati (NumPy, pandas) e per la produzione dei grafici (Matplotlib, Seaborn). Le opzioni di visualizzazione di pandas sono configurate in modo da garantire la corretta rappresentazione di tutte le colonne e righe durante le ispezioni intermedie del DataFrame.

In [ ]:
#Importazione pacchetti necessari per l'analisi
import numpy as np   # Serve a caricare la libreria NumPy
import pandas as pd   # Serve a caricare la libreria Pandas
import seaborn as sns   # Serve a caricare la libreria Seaborn 
import matplotlib.pyplot as plt   # Serve a caricare il modulo "pyplot" della libreria Matplotlib
import matplotlib.patches as mpatches   # Serve a caricare il modulo "patches" della libreria Matplotlib

pd.set_option('display.max_rows', None)   # Costringe python a mostrare tutte le righe del dataset
pd.set_option('display.max_columns', None)   # Costringe python a mostrare tutte le righe del dataset
pd.set_option("display.max_info_columns", 200) # Agisce sul comando .info() in modo che mostri un riepilogo completo di tutte le colonne

## Caricamento del dataset

Il dataset di riferimento è costituito dal file `2025_data.csv`, contenente le statistiche di tutte le partite professionistiche di *League of Legends* disputate nel 2025. L'analisi che segue è circoscritta alla *League of Legends EMEA Championship* (LEC).

In [ ]:
#Importazione file .csv contenente i dati
df = pd.read_csv('2025_data.csv', low_memory=False)

In [ ]:
df.head(12)   # Mostra le prime 12 righe di un dataset

### Conversione dei tipi di dato

Per ottimizzare l'utilizzo della memoria e garantire la corretta interpretazione delle variabili, si applica la conversione automatica dei tipi tramite `convert_dtypes()`. Successivamente vengono forzate le conversioni delle colonne temporali e booleane.

In [ ]:
df2 = df.convert_dtypes()   # Analizza ogni colonna e cerca di convertirla nel tipo di dato più appropriato e moderno possibile


In [ ]:
df = df2

In [ ]:
#===============================================

cols = ['playoffs','firstPick','result','firstdragon','firstherald','firstbaron','atakhans','opp_atakhans','firsttower', 'firstmidtower','firsttothreetowers']

# Si crea una lista di colonne da utilizzare come filtro

#===============================================
df['date'] = df['date'].astype('datetime64[ns]')   # Converte la colonna 'date' nel formato dati 'datetime64[ns]'
df[cols] = df[cols].astype('boolean')   # Applica la lista 'cols' come filtro per convertire quelle colonne nel formato dati 'boolean'
df.info()   # Mostra le informazioni relative alle colonne df

# 1. Analisi generali — Contesto competitivo

Questa prima sezione fornisce una caratterizzazione complessiva del campionato LEC 2025, analizzando il numero di partite disputate, la loro distribuzione tra gli split competitivi, la durata media e l'impatto degli elementi strutturali del gioco (side, first blood, primo Baron, obiettivi maggiori) sull'esito finale.

## 1.1 Numero di partite disputate per split

**Quante partite sono state giocate nel LEC nel 2025?**

In [ ]:
df['league'].unique()   # Restituisce i valori unici della colonna 'league'

In [ ]:
lec_df = df[df['league'] == 'LEC']   # Filtra il dataframe df mantenendo solo le righe con 'LEC' nella colonna
                                     # 'league' e assegna il risultato a lec_df."

lec_df['split'].unique()   # Restituisce i valori unici della colonna 'split' del dataframe 'lec_df'

Nel dataset del LEC 2025 sono presenti i tre split competitivi: **Winter**, **Spring** e **Summer**.

In [ ]:
games_per_split = (                     # Crea il dataframe 'games_per_split' eseguendo una concatenazione in Pandas
    lec_df                              # Quì è indicato il df di partenza (in questo caso 'lec_df')
    .drop_duplicates(subset='gameid')   # Questa riga rimuove tutti i duplicati della riga 'gameid' 
    .groupby('split')['gameid']         # Raggruppa le righe per valore di split e seleziona la colonna gameid per il conteggio delle partite
    .count()                            # Esegue il conteggio dei 'gameid' per ogni split
)
total_games = games_per_split.sum()     # Esegue la somma dei valori di 'games_per_split'

print(games_per_split)                  # Stampa il df 'games_per_split'
print(total_games)                      # Stampa il valore 'total_games'

### Osservazione grafico

Durante la stagione 2025 sono state disputate complessivamente **306 partite** nel LEC, così ripartite:

- **Winter Split**: 84 partite
- **Spring Split**: 139 partite
- **Summer Split**: 83 partite

Poiché ogni partita è identificata univocamente da `gameid` ma compare più volte nel dataset per rappresentare le statistiche di team e di singolo giocatore, il conteggio è stato effettuato sui soli valori unici della colonna `gameid`. Le partite sono state quindi aggregate per split, ottenendo la distribuzione delle partite lungo la stagione competitiva.

In [ ]:
summary_gamesPerSplit_df = games_per_split.reset_index()   # Crea il df 'summary_gamesPerSplit' partendo da 'games_per_split' e resetta
                                                           # gli indici

summary_gamesPerSplit_df.columns = ['split', 'games']      # Rinomina tutte le colonne del DataFrame assegnando i nuovi nomi nell’ordine 
                                                           # in cui compaiono

total_games = summary_gamesPerSplit_df['games'].sum()      # Crea la variabile 'total_games' sommando i valori della colonna 'games'
                                                           # appartenente al df 'summary_gamesPerSplit'
                                                           # NOTA BENE: lo stesso calcolo era stato effettuato nel blocco di codice
                                                           # precedente utilizzando anche lo stesso nome per la variabile è quindi
                                                           # un passaggio aggiuntivo non necessario in quanto si poteva utilizzare la
                                                           # vecchia variabile

summary_gamesPerSplit_df['percentage (%)'] = (             # Riga di codice che inizializza la colonna 'percentage (%)' nel df
                                                           # 'summary_gamesPerSplit'
    
    summary_gamesPerSplit_df['games'] / total_games * 100  # Divide ogni valore della colonna 'games' per il valore di 'total_games'
                                                           # e successivamente moltiplica il risultato per 100
    
).round(2)                                                 # Arrotonda il risultato dell'operazione precedente a due cifre decimali           

summary_gamesPerSplit_df.head()                            # Mostra le prime righe del df 'summary_gamesPerSplit'
                                                           # NOTA BENE: In questo caso sono solamente tre righe perchè il df non ne contiene
                                                           # altre

In [ ]:

labels = summary_gamesPerSplit_df['split']   # Assegna la colonna "split" del df summary_gamesPerSplit alla variabile "labels"
sizes = summary_gamesPerSplit_df['games']    # Assegna la colonna "games" del df summary_gamesPerSplit alla variabile "sizes"

fig, ax = plt.subplots(figsize=(7, 7))   # .subsplot restituisce gli oggetti "Figure" e "Axes" assegnati rispettivamente alle
                                         # variabili "fig" e "ax". 
                                         # NOTA BENE: "figsize" è un parametro di "Figure" passato a ".subsplot" tramite **fig_kw 

ax.pie(                                  # ".pie" è un metodo dell'ogetto "Axes" utilizzato per creare il grafico a torta
    
    sizes,                               # Nella documentazione ufficiale questo è il parametro "x : 1D array-like" e indica le informazioni
                                         # che verranno utilizzate per costruire il grafico. In questo caso x = sizes, cioè la variabile
                                         # creata all'inizio
    
    labels=labels,                       # Etichette testuali associate a ciascuna fetta del grafico
    
    autopct='%1.1f%%',                   # Mostra le percentuali dentro le fette.
                                         # Può avere tre valori:
                                         # None: Non mostra nulla
                                         # Stringa di formato: Visualizza la percentuale formattata
                                         # (es. "%1.1f%%" mostra una percentuale con una cifra decimale)
                                         # Funzione: Viene chiamata una funzione per ricevere un'etichetta personalizzata
    
    startangle=90,                       # Angolo da cui il grafico inizia muovendosi poi in senso anti orario, in questo esempio
                                         # specifico startangle=90 farà partire il grafico dalle ore 12.
    
    wedgeprops={'width': 0.4},           # Applica impostazioni estetiche ad ogni fetta del grafico. In questo caso width: 0.4
                                         # scava al centro rendendo il grafico a torta un grafico a ciambella
    
    pctdistance=0.80                     # Distanza relativa al raggio lungo la quale il testo di "autopct" viene generato.
                                         # se si vuole inserire il testo fuori dal grafico impostare "pctdistance" > 1
)

# Numero totale al centro
ax.text(                                 # ".text" è il metodo utilizzato per inserire il testo nel grafico
    
    0, 0,                                # Queste sono le coordinate rispettivamente x ed y che è possibile modificare per muovere
                                         # il testo nella figura. In questo caso sono entrambe 0 quindi il testo è perfettamente centrato
    
    f'{total_games}\nTotal Games',       # Parametro text, utilizzato per inserire la stringa da visualizzare, in questo caso
                                         # è una f-string che inserisce il valore della variabile "total_games" e, tramite /n
                                         # va a capo in modo da creare un'etichetta su due righe

    ha='center',                         # Il parametro ha sta per "horizontalalignment", allinea orizzontalmente il testo 
                                         # rispetto al punto (x, y), in questo caso 'center' lo centra orizzontalmente sul punto specificato

    va='center',                         # Il parametro va sta per "verticalalignment", allinea verticalmente il testo rispetto
                                         # al punto (x, y), in questo caso 'center' lo centra verticalmente sul punto specificato

    fontsize=12,                         # Parametro che regola la grandezza del testo

    fontweight='bold'                    # Parametro che regola lo spessore del testo
)

ax.set_title(                                           # Parametro utilizzato per impostare il titolo del grafico
    
    'Distribuzione delle partite LEC 2025 per Split',   # Parametro label nella documentazione ufficiale, indica la stringa di testo che
                                                        # verrà mostrata come titolo
    
    fontsize=14                                         # Parametro che regola la grandezza del titolo.
                                                        # NOTA BENE: nella documentazione ufficiale non comprare "fontsize" tra i parametri
                                                        # del metodo ".set_title()", compare però il parametro "**kwargs" che serve
                                                        # per utilizzare le parole chiave relative alle impostazioni di testo
)

plt.savefig('figures/distribuzione_partite_per_split.png', dpi=150, bbox_inches='tight')
plt.show()                                              # Metdo utilizzato per visualizzare tutti i grafici creati e non ancora mostrati


## 1.2 Durata media delle partite

**Qual è stata la durata media delle partite LEC nel 2025?**

In [ ]:
games_duration_df = lec_df.drop_duplicates(subset='gameid').copy()   # Creazione del df "games_duration_df" a partire dal df "lec_df"

games_duration_df['gamelength_min'] = games_duration_df['gamelength'] / 60   # Creazione della colonna 'gamelength_min' nel df
                                                                             # 'games_duration_df' che si ricava dalla divisione dei valori
                                                                             # della colonna 'gamelength' (dello stesso df) per 60

mean_duration = games_duration_df['gamelength_min'].mean()                   # Variabile che contiene il valore della media della durata
                                                                             # (in min) della lunghezza delle partite

median_duration = games_duration_df['gamelength_min'].median()               # Variabile che contiene il valore della mediana della durata
                                                                             # (in min) della linghezza delle partite

n_total = games_duration_df['gamelength_min'].shape[0]                       # Conta il numero di righe che ci sono nella colonna
                                                                             # 'gamelength_min'.
                                                                             # .shape() restituisce una tupla con il numero di righe 
                                                                             # e colonne ma in questo caso essendo .shape[0] restituisce 
                                                                             # solamente il numero delle righe

n_over_45 = (games_duration_df['gamelength_min'] > 45).sum()                 # Restituisce il numero di partite di durata superiore a 45 min

pct_over_45 = n_over_45 / n_total * 100                                      # Calcola la percentuale di partite che durano più di 45 min

In [ ]:
plt.figure(figsize=(8, 5))   # Crea una nuova figura matplotlib impostando la dimensione (in pollici) della tela

ax = sns.histplot(games_duration_df['gamelength_min'], bins=20, kde=True)   # Inizializza l'oggetto Axes di matplotlib direttamente
                                                                            # in seaborn con la creazione di un istogramma.

                                                                            # Il primo parametro indica i dati utilizzati per il grafico
                                                                            # Il parametro "bins" indica il numero di intervalli in cui si
                                                                            # divide il grafico

                                                                            # Il parametro "kde" può assumere i valori "True" o "False"
                                                                            # e dice a seaborn se tracciare o no la curva che stima la
                                                                            # distribuzione dei dati

                                                                            # NOTA BENE:
                                                                            # In questo esempio viene utilizzato un metodo implicito, per
                                                                            # rendere il codice più chiaro converrebbe inizializzare
                                                                            # gli oggetti Figure ed Axes tramite il comando ".subsplot"
                                                                            # di matplotlib

# Linee verticali: media e mediana
ax.axvline(mean_duration, linestyle='--', linewidth=2, label=f"Media: {mean_duration:.1f} min")
ax.axvline(median_duration, linestyle=':', linewidth=2, label=f"Mediana: {median_duration:.1f} min")
# Metodo dell'oggetto "Axes" che permette di tracciare una linea verticale sul grafico
# "mean_duration" rappresenta la variabile "x" nella documentazione ufficiale, è il punto sull'asse delle "x" in cui la linea viene tracciata
# "linestyle" è un parametro grafico, serve per impostare lo stile grafico della linea
# "linewidth" è un parametro grafico, serve per impostare lo spessore della linea
# "label" parametro che serve per inserire l'etichetta che verrà visualizzata nella legenda

# Asse Y con step fisso (come hai già fatto)
y_max = ax.get_ylim()[1]                    # Il metodo ".get_ylim" serve ad ottenere i limiti dell'asse delle y, il metodo restituisce
                                            # la tupla [y_min, y_max] che rappresentano i valori rispettivamente del limite minimo e massimo.
                                            # In questo caso .get_ylim()[1]" serve ad ottenere solamente il limite massimo dell'asse delle y

ax.set_yticks(np.arange(0, y_max + 1, 5))   # Serve per impostare manualmente gli intervalli (i ticks) sull'asse delle y
                                            # "np.arange(0, y_max + 1, 5)" serve a creare un intervallo di numeri che vanno da 0 a y_max+1
                                            # con un intervallo di 5 tra un numero e l'altro. 
                                            # "np.arange(0, y_max + 1, 5)" viene poi passato come parametro "ticks" della documentazione
                                            # ufficiale

# Griglia orizzontale
ax.set_axisbelow(True)                         # Metodo che gestisce se la griglia della tavola viene tracciata sopra il grafico

ax.grid(axis='y', linestyle='--', alpha=0.6)   # Metodo che si utilizza per configurare la griglia del grafico
                                               # Il parametro "axis" serve ad impostare l'asse su cui la griglia viene applicata
                                               # Il parametro "linestyle" serve per impostare lo stile grafico della griglia
                                               # Il parametro "alpha" controlla la trasparenza della griglia (i valori vanno da 0 a 1)

# Titoli e label
ax.set_xlabel('Durata partita (minuti)')                              # Il metodo "set_xlabel" serve ad impostare l'etichetta sull'asse
                                                                      # delle x

ax.set_ylabel('Numero di partite')                                    # Il metodo "set_ylabel" serve ad impostare l'etichetta sull'asse
                                                                      # delle y

ax.set_title('Distribuzione della durata delle partite – LEC 2025')   # Il metodo "set_title" serve ad impostare il titolo del grafico

# Box testuale con supporto quantitativo (>45 min)
info = (                                            # Variabile "info" che contiene una f-string formattata su più righe
    f"Totale partite: {n_total}\n"                  # che costituisce una sezione della legenda del grafico
    f"> 45 min: {n_over_45} ({pct_over_45:.1f}%)"   
)

ax.text(                                            # Metodo dell'oggetto "Axes" utilizzato per inserire del testo sul grafico
                                                    # in questo caso viene utilizzato per inserire le informazioni relative alla variabile
                                                    # info
    
    0.98, 0.95, info,                               # 0.98 e 0.95 rappresentanto le variabili x ed y nella documentazione ufficiale 
                                                    # e servono a spostare il testo sulla tavola, "info" rappresenta la variabile "s"
                                                    # nella documentazione, cioè la stringa di testo da inserire nel grafico
    
    transform=ax.transAxes,                         # Il parametro "transform" dice come interpretare le coordinate x, y dell'oggetto a cui
                                                    # è assegnato. In questo caso "ax.transAxes" fa in modo che x ed y vengano interpretati
                                                    # come posizioni relative dentro al grafico (il testo essendo vicino al punto 
                                                    # di coordinate 1, 1 viene impostato nell'angolo in alto a dx della tavola)
    
    ha='right',                                     # Imposta l'allineamento orizzontale del testo rispetto al punto (x, y)
    
    va='top',                                       # Imposta l'allineamento verticale del testo rispetto al punto (x, y)
    
    bbox=dict(                                      # Il parametro "bbox" definisce le proprietà del riquadro che contiene il testo.
                                                    # "dict()" crea un dizionario di proprietà utilizzate per configurare il box
        
        boxstyle='round',                           # "boxstyle" definisce la forma del riquadro, in questo caso "round" specifica 
                                                    # che il riquadro deve aver ei bordi arrotondati
        
        alpha=0.2                                   # Imposta l'opacità normalizzata del riquadro (0.2 viene interpretato come percentuale)
    )          
                                    
)

ax.legend()                                         # Aggiunge una legenda al grafico utilizzando le label degli elementi disegnati
                                                    # in questo caso "Media" e "Mediana"

plt.savefig('figures/distribuzione_durata_partite.png', dpi=150, bbox_inches='tight')
plt.show()                                          # Metdo utilizzato per visualizzare tutti i grafici creati e non ancora mostrati


### Osservazione grafico

La durata media delle partite del LEC 2025 risulta pari a circa **33 minuti**. La distribuzione è tuttavia influenzata dalla presenza di 15 partite di durata particolarmente elevata (oltre 45 minuti), che contribuiscono a innalzare il valore medio. La maggior concentrazione di osservazioni si colloca nell'intervallo compreso approssimativamente tra 27 e 32 minuti, suggerendo che la durata tipica di una partita risulti inferiore alla media complessiva.

Combinando tale evidenza con il valore medio di circa 27 kill per partita (approfondito nella sezione dedicata), si ricava un ritmo di gioco di circa 0,8 kill al minuto. Tali risultati delineano un campionato dalle caratteristiche prevalentemente *early–mid game oriented*, con un ritmo sostenuto e una frequente interazione tra le squadre già nelle fasi iniziali della partita.

## 1.3 Durata delle partite per split

**La durata media delle partite LEC è variata nei vari split?**

In [ ]:
summary_split_df = (                         # Inizializza la variabile che conterrà il nuovo df
    
    games_duration_df                        # Df di partenza da cui si manipoleranno i dati per creare quello nuovo
    
    .groupby('split')['gamelength_min']      # ".groupby('split')" raggruppa i dati in base ai valori unici della colonna "split"
                                             # "["gamelength_min"]" stabilisce la colonna su cui effettuare l'operazione
    
    .agg(                                    # .agg() applica una o più operazioni ai dati raggruppati dal .groupby() precedente
                                             # La sintassi per questa applicazione è la seguente nome_della_colonna = "funzione_ufficiale"
        
        media='mean',                        # Crea la colonna "media" che conterrà la media della durata delle partite per ogni split
        
        mediana='median',                    # Crea la colonna "mediana" che conterrà il valore centrale della durata delle partite per
                                             # ogni split
        
        std='std',                           # Crea la colonna "std" che conterrà la deviazione standard, ovvero quanto le durate variano
                                             # attorno alla media per ogni split
        
        min='min',                           # Crea la colonna "min" che conterrà il valore minimo di durata delle partite per ogni split
        
        max='max',                           # Crea la colonna "max" che conterrà il valore massimo di durata delle partite per ogni split
        
        partite='count'                      # Crea la colonna "partite" che conterrà il conteggio delle partite per ogni split
    )
    .round(2)                                # Il metodo ".round(2)" arrotonda i valori decimali a due cifre dopo la virgola
)

In [ ]:
summary_split_df.head()   # Mostra le prime 5 righe del dataframe "summary_split_df"

In [ ]:
plt.figure(figsize=(8, 5))     # Crea una nuova figura matplotlib impostando la dimensione (in pollici) della tela

ax = sns.boxplot(              # Seaborn crea e inizializza l'oggetto "Axes" salvandolo in ax, con la creazione di un boxplot
    
    data=games_duration_df,    # Nella documentazione ufficiale "data" rappresenta il DataFrame di riferimento per la creazione del grafico
                               # in questo caso viene utilizzato il DataFrame "games_duration_df"
    
    x='split',                 # Il parametro "x" nella documentazione ufficiale indica i valori che verranno utilizzati
                               # sull'asse delle x, in questo caso i valori della colonna 'split'
    
    y='gamelength_min'         # Il parametro "y" nella documentazione ufficiale indica i valori che verranno utilizzati
                               # sull'asse dell y, in questo caso i valori della colonna 'gamelength_min'
)

plt.xlabel('Split')            # 'plt.xlabel()' imposta l'etichetta testuale dell'asse delle x, in questo caso "split"

plt.ylabel('Durata partita (minuti)')   # plt.ylabel() imposta l'etichetta testuale dell'asse delle y, 
                                        # in questo caso "Durata partita (minuti)"

plt.title('Distribuzione della durata delle partite per Split – LEC 2025')   # 'plt.title()' imposta il titolo del grafico

plt.savefig('figures/boxplot_durata_partite_per_split.png', dpi=150, bbox_inches='tight')
plt.show()   # plt.show() renderizza e visualizza tutti i grafici costruiti ma non ancora mostrati fino a questo punto del codice


### Osservazione grafico

Dai boxplot emerge che la mediana della durata delle partite risulta sostanzialmente simile nei tre split, indicando che la durata tipica delle partite non varia in modo significativo tra Winter, Spring e Summer. Lo split Winter presenta tuttavia una maggiore variabilità, evidenziata da un intervallo interquartile più ampio e dalla presenza di valori estremi più distanti rispetto agli altri due split. Sebbene le mediane risultino comparabili, le durate medie mostrano lievi differenze tra gli split, suggerendo che la variabilità osservata — particolarmente accentuata nello split Winter — contribuisca a influenzare il valore medio complessivo.

## 1.4 Win rate Blue side vs Red side

**Qual è la percentuale di vittorie Blue side rispetto al Red side nel LEC 2025?**

In [ ]:
df_side = lec_df.loc[lec_df['position'] == 'team', ['gameid', 'split', 'side', 'result']].copy()
# Si crea il DataFrame 'df_side' utilizzando il DataFrame 'lec_df' come DataFrame di partenza
# La funzione '.loc' filtra le righe del DataFrame 'lec_df' mantenendo solamente le righe che hanno come valore 'team' nella colonna 'position'
# e successivamente esplicita le colonne da tenere del DataFrame di partenza
# .copy() crea una copia indipendente del risultato, evitando che modifiche a 'df_side' influenzino 'lec_df'

df_side.head(10)   # Mostra le prime 10 righe del DataFrame 'df_side'

In [ ]:
df_side.groupby('gameid')['side'].nunique().value_counts() # Serve a verificare che ogni partita (gameid) abbia esattamente 2 side
# df_side è il DataFrame a cui si applicano le funzioni seguenti
# ".groupby('gameid')" raggruppa i dati in base ai valori unici della colonna "gameid"
# "["side"]" stabilisce la colonna su cui effettuare le operazioni
# ".nunique()" conta il numero di valori unici presenti nella colonna "side"
# ".value_counts()" conta quante volte appare ogni risultato di ".nunique()"

In [ ]:
summary_side_split = (                    # Nome del DataFrame che si sta creando
    
    df_side                               # DataFrame di partenza
    
    .groupby(['split', 'side'])           # Colonne su cui raggruppare i dati, in questo caso essendo due colonne il raggruppamento verra 
                                          # effettuato utilizzando tutte le combinazioni dei valori delle colonne 
    
    .agg(                                 # Applica una o più operazioni ai dati raggruppati dal groupby precedente
                                          # In questo caso si utilizza la sintassi "Named Aggregation": nome_colonna = ('colonna_sorgente', 'funzione')
        
        partite=('gameid', 'nunique'),    # Crea la colonna 'partite' contando i valori unici della colonna 'gameid'
        vittorie=('result', 'sum')        # Crea la colonna 'vittorie' sommando i valori booleani della colonna 'result' (vittoria=1, sconfitta=0)
    )
    .reset_index()                        # Dopo il .groupby() le colonne 'split' e 'side' diventano l'indice del DataFrame,
                                          # .reset_index() le riporta come colonne normali e reimposta l'indice con numeri progressivi
)


summary_side_split['win_rate'] = (        # Crea la colonna "win_rate" nel DataFrame "summary_side_split" 
                                          # effettuando il rapporto tra la colonna 'vittorie' e la colonna 'partit
    
    summary_side_split['vittorie'] /
    summary_side_split['partite']
    
).round(2)                                # Arrotonda alla seconda cifra decimale il risultato della colonna

In [ ]:
summary_side_split.head(10)   # Mostra le prime 10 colonne del DataFrame "summary_side_split"

In [ ]:
plt.figure(figsize=(8, 5))                                           # Crea ed inizializza l'oggetto "figure", cioè la tavola o tela su cui 
                                                                     # verrà poi disegnatgo il grafico tramite matplotlib

ax = sns.barplot(                                                    # Crea ed inizializza l'oggetto axes 
                                                                     # tramite la creazione del grafico a barre con Seaborn
    
    data=summary_side_split,                                         # 'data' rappresenta il parametro che conterrà i dati con cui creare il grafico, 
                                                                     # "summary_side_split" è il dataframe di riferimento che contiene i dati 
                                                                     # da utilizzare
    
    x='split',                                                       # 'x' è il parametro che controlla quali dati mettere a grafico sull'asse delle x, 
                                                                     # "split" è la colonna del DataFrame da cui verranno prelevati i dati
    
    y='win_rate',                                                    # 'y' è il parametro che controlla quali dati mettere a grafico sull'asse delle y, 
                                                                     # "win_rate" è la colonna del DataFrame da cui verranno prelevati i dati
    
    hue='side'                                                       # Il parametro "hue" suddivide ogni gruppo dell'asse x in sottogruppi
                                                                     # assegnando un colore diverso ad ognuno, in questo caso divide
                                                                     # ogni split in Blue e Red permettendo il confronto visivo tra i due side
)

ax.set_ylabel('Win rate')                                            # Imposta l'etichetta dell'asse delle y, in questo caso "Win rate'

ax.set_title('Win rate Blue vs Red per Split – LEC 2025')            # Imposta il titolo del grafico

ax.set_ylim(0, 1)                                                    # Imposta i valori minimo e massimo che possono comparire sull'asse delle y
                                                                     # in questo caso siccome si tratta di percentuali valori vanno da 0 e 1

# ✅ aggiunta etichette percentuali
for container in ax.containers:                                      # l'attributo '.containers' crea una tupla di oggetti (Bar Container)  
                                                                     # relativa alle barre create tramite il parametro 'hue', 
                                                                     # quindi in questo caso restituirà una lista tipo (red, blue). 
                                                                     # Il ciclo for itrea ogni container (quindi ogni valore della lista)
    
    ax.bar_label(                                                    # Aggiunge le etichette alle barre di un determinato Bar Container
        
        container,                                                   # Questo parametro indica su quale Bar Container applicare le etichette.
                                                                     # Ad ogni iterazione del loop la variabile assume il valore
                                                                     # del container corrente (in questo caso prima Blue e poi Red)
        
        labels=[f'{v*100:.1f}%' for v in container.datavalues],      # Il parametro 'labels' indica la stringa che verrà utilizzata come etichetta.
                                                                     # Si tratta di una f-string che prende la variabile 'v' e la moltiplica per 100,
                                                                     # arrotonda ad una cifra decimale ed aggiunge il simbolo '%' alla fine.
                                                                     # La variabile 'v' viene creata attraverso '.datavalues' che restituisce i valori
                                                                     # 'win_rate' che determinano l'altezza delle barre, stabiliti a monte dal parametro
                                                                     # y = 'win_rate' nella creazione del grafico
        
        padding=3,                                                   # Il parametro "padding" rappresenta la distanza in punti
                                                                     # tra la fine della barra e l'etichetta, in questro caso 3 punti
        
        fontsize=10                                                  # Il parametro "fontsize" regola la dimensione del testo delle etichette,
                                                                     # in questo caso impostata a 10 punti
    )

plt.savefig('figures/winrate_blue_vs_red_per_split.png', dpi=150, bbox_inches='tight')
plt.show()                                                           # 'plt.show()' renderizza e visualizza tutti i grafici costruiti 
                                                                     # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

Per l'analisi del win rate per side sono state considerate esclusivamente le righe a livello di team, in modo da includere entrambi i side per ciascuna partita. Nello **Spring Split** emerge un netto vantaggio del Blue side, con una percentuale di vittorie superiore di circa 20 punti percentuali rispetto al Red side. Nel **Winter Split** le differenze risultano più contenute, suggerendo un sostanziale equilibrio tra i due side. Nel **Summer Split** si osserva una lieve inversione di tendenza, con un win rate del Red side marginalmente superiore a quello del Blue side, sebbene la differenza risulti limitata.

### 1.4.1 Approfondimento: win rate per fascia di durata e side

Per verificare se la durata della partita influisca sulla differenza di win rate tra Blue side e Red side, le partite vengono suddivise in sei fasce basate sui quantili della distribuzione della durata, e per ciascuna fascia viene calcolato il win rate dei due side.

In [ ]:
df_winsidelength = (                                                                  # Inizializzazione del nuovo DataFrame che verrà chiamato 
                                                                                      # "df_winsidelength"
    
    lec_df                                                                            # DataFrame di partenza da cui verranno filtrate le colonne
                                                                                      # che comporranno il nuovo DataFrame
    
    .loc[lec_df['position'] == 'team', ['gameid', 'side', 'gamelength', 'result']]    # '.loc' con il primo argomento filtra le righe 
                                                                                      # mantenendo solo quelle che nella colonna 'position' hanno 
                                                                                      # come valore 'team', mentre con il secondo argomento indica
                                                                                      # quali colonne mantenere da DataFrame di partenza, selezionando
                                                                                      # solo i dati rilevanti per l'analisi
    
    .copy()                                                                           # '.copy()' crea una copia indipendente del risultato, evitando 
                                                                                      # che le modifiche a 'df_winsidelength' influenzino 'lec_df'
)

df_winsidelength['gamelength_min'] = df_winsidelength['gamelength'] / 60              # Crea la colonna 'gamelength_min' nel DataFrame 'df_winsidelength'
                                                                                      # dividendo i valori della colonna 'gamelength' per 60, 
                                                                                      # convertendo la durata da secondi a minuti

df_winsidelength['duration_bin'] = pd.qcut(                                           # Crea la colonna 'duration_bin' nel DataFrame 'df_winsidelenght'
                                                                                      # attraverso la funzione pandas .qcut()
                                                                                      # La funzione '.qcut()' serve a dividere i dati in quantili, 
                                                                                      # cioè intervalli con lo stesso numero di osservazioni.
                                                                                          
    df_winsidelength['gamelength_min'],                                               # 'df_winsidelength['gamelength_min']' rappresenta 
                                                                                      # il parametro x nella documentazione ufficiale, ossia,
                                                                                      # l'array di dati su cui la funzione andrà ad operare
    
    q=6,                                                                              # Il parametro 'q' indica il numero di intervalli (quantili) 
                                                                                      # da generare, in questo caso per q = 6 verranno generati 
                                                                                      # 6 intervalli (o quantili)
    
    precision = 2                                                                     # Il parametro 'precision' regola la precisione decimale con cui
                                                                                      # vengono mostrati i valori degli intervalli nelle etichette
                                                                                      # in questo caso per precision = 2 i valori verranno arrotondati
                                                                                      # alla seconda 
)


In [ ]:
winrate_by_len_side = (                                    # Creazione del DataFrame 'win_rate_by_len_side'
    
    df_winsidelength                                       # DataFrame di partenza da cui verranno elaborati i dati
    
    .dropna(subset=['duration_bin'])                       # Elimina le righe che contengono valori nulli nella colonna 'duration_bin'
    
    .groupby(['duration_bin', 'side'], observed = True)    # Raggruppa i dati per i valori dlle colonne 'duration_bin' e 'side'
                                                           # Il parametro 'observed=True' è necessario perché 'duration_bin' è di tipo
                                                           # 'Categorical' (creato da pd.qcut), e indica di mostrare solo gli intervalli
                                                           # che contengono effettivamente dati, escludendo quelli vuoti
    
    .agg(                                                  # .agg() applica la sintassi "Named Aggregation" 
                                                           # nome_colonna = ('colonna_sorgente', 'funzione').
        
        partite=('gameid', 'nunique'),                     # Crea la colonna 'partite' contando i valori unici della colonna 'gameid'
        
        win_rate=('result', 'mean')                        # Crea la colonna 'win_rate' calcolando la media della colonna 'result'
                                                           # che essendo booleana (vittoria=1, sconfitta=0) restituisce direttamente il win rate
    )
    .reset_index()                                         # Dopo il .groupby() le colonne 'partite' e 'win_rate' diventano l'indice del DataFrame,
                                                           # .reset_index() le riporta come colonne normali e reimposta l'indice con numeri progressivi
)         

winrate_by_len_side.head(12)                               # # Mostra le prime 12 righe del DataFrame 'winrate_by_len_side'

In [ ]:
plt.figure(figsize=(10, 5))       # Inizializza la tavola su cui verrà disegnato il grafico, figsize imposta la grandezza della tavola

ax = sns.barplot(                 # Inizializza l'ogetto 'Axes' tramite la creazione di un grafico a barre in seaborn
                    
    data=winrate_by_len_side,     # Parametro che indica i dati da utilizzare
    
    x='duration_bin',             # Parametro che identifica quale colonna di "data" verrà utilizzata sull'asse delle "x"
    
    y='win_rate',                 # Parametro che identifica quale colonna di "data" verrà utilizzata sull'asse delle "y"
    
    hue='side'                    # Parametro che serve colorare le barre, in questo caso dato che 'side' ha due valori, 
                                  # oltre al colore crea due barre per ogni dato nell'asse delle "x"
)
ax.set_ylim(0, 1)                 # Imposta i limiti dei valori massimi e minimi che la scala sull'asse delle "y" può avere

plt.xticks(rotation=45)           # Ruota di 45° le etichette che che identificano i dati sull'asse delle "x", in questo caso specifico
                                  # serve a non far sovrapporre i nomi degli intervalli tra loro

plt.tight_layout()                # Aggiusta automaticamente i margini della figura per evitare che gli elementi del grafico 
                                  # (titolo, etichette, assi) si sovrappongano o vengano tagliati

for container in ax.containers:   # L'attributo '.containers' crea una tupla di oggetti (Bar Container)
                                  # relativa alle barre create tramite il parametro 'hue',
                                  # quindi in questo caso restituirà una lista tipo (red, blue). 
                                  # Il ciclo for itrea ogni container (quindi ogni valore della lista)
    
    ax.bar_label(                 # Aggiunge le etichette alle barre di un determinato Bar Container
        
        container,                # Questo parametro indica su quale Bar Container applicare le etichette.
                                  # Ad ogni iterazione del loop la variabile assume il valore
                                  # del container corrente (in questo caso prima Blue e poi Red) 
        
        labels=[f'{v*100:.1f}%' for v in container.datavalues],     # Il parametro 'labels' indica la stringa che verrà utilizzata 
                                                                    # come etichetta. Si tratta di una f-string che prende la variabile 'v' 
                                                                    # e la moltiplica per 100, arrotonda ad una cifra decimale ed 
                                                                    # aggiunge il simbolo '%' alla fine. La variabile 'v' viene creata 
                                                                    # attraverso '.datavalues' che restituisce i valori 'win_rate' che determinano 
                                                                    # l'altezza delle barre, stabiliti a monte dal parametro y = 'win_rate' 
                                                                    # nella creazione del grafico
        
        padding=3,      # Il parametro "padding" rappresenta la distanza in punti
                        # tra la fine della barra e l'etichetta, in questro caso 3 punti
        
        fontsize=10     # Il parametro "fontsize" regola la dimensione del testo delle etichette,
                        # in questo caso impostata a 10 punti
    )

ax.set_title('Win rate Blue vs Red 2025 per Fascia di durata')     # Imposta il titolo del grafico
    
plt.savefig('figures/winrate_blue_vs_red_per_fascia_durata.png', dpi=150, bbox_inches='tight')
plt.show()              # 'plt.show()' renderizza e visualizza tutti i grafici costruiti 
                        # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

Per valutare se la durata delle partite influisca sul win rate dei due side, sono state considerate esclusivamente le righe a livello di team, in modo da ottenere due osservazioni per ciascuna partita (Blue e Red). La durata è stata convertita in minuti e suddivisa in sei fasce basate sui quantili della distribuzione, così da ottenere gruppi di numerosità comparabile, sui quali è stato calcolato il win rate per side.

Dal grafico emerge che in 5 fasce su 6 il Blue side presenta un win rate superiore, generalmente con un margine ben definito. Fa eccezione la fascia **28,55–30,56 minuti**, nella quale il vantaggio del Blue side risulta più contenuto (circa 4 punti percentuali). Nella fascia di durata più elevata si osserva invece un'inversione, con il Red side in vantaggio di circa 6 punti percentuali.

## 1.5 Numero medio di kill per partita

**Qual è il numero medio di kill per partita nel LEC 2025?**

In [ ]:
killspergame_df = lec_df[lec_df['position'] == 'team'][['gameid', 'teamkills', 'teamdeaths']]
# Crea il dataframe 'killspergame_df' partendo da lec_df filtrando tutte le righe che hanno il valore 'team' nella colonna 'position' 
# mantenendo nel risultato finale solamente le colonne 'game_id', 'teamkills' e 'teamdeaths'

killspergame_df = killspergame_df.drop_duplicates(subset=['gameid'])   # '.drop_duplicates()' elimina le righe duplicate nella 
                                                                       # colonna 'gameid', mantenendo una sola riga per ogni partita

killspergame_df['total_kills'] = killspergame_df['teamkills'] + killspergame_df['teamdeaths']
# Crea la colonna 'total_kills' i cui valori sono la somma delle kill e delle morti del team, ottenendo così il totale delle kill per partita

average_kills = killspergame_df['total_kills'].mean()   # Crea la variabile 'average_kills' e salva al suo interno la media di 'total_kills'

# Impostiamo lo stile
sns.set_style("ticks")          # Imposta lo stile estetico del grafico, 
                                # "ticks" è uno degli stili disponibili in Seaborn che mostra i segni di graduazione
                                # sugli assi rimuovendo la griglia di sfondo

plt.figure(figsize=(12, 6))     # Inizializza l'oggetto 'figure' che rappresenta la 'tavola' su cui poi verrà renderizzato il grafico 

sns.histplot(killspergame_df['total_kills'], kde=True, color="teal", bins=12)
# Crea un istogramma usando i valori della colonna 'total_kills' come dati.
# 'kde=True' calcola e mostra la linea di densità per visualizzare meglio la distribuzione
# 'color="teal"' imposta il colore delle barre del grafico
# 'bins=12' regola il numero di barre (intervalli) da mostrare


plt.axvline(average_kills, color='red', linestyle='--', linewidth=2, label=f'Media: {average_kills:.2f}')
# 'plt.axvline()' disegna una linea verticale sull'asse delle x
# in corrispondenza del valore 'average_kills' (la media delle kill totali)
# 'color="red"' imposta il colore della linea
# 'linestyle="--"' imposta lo stile della linea come tratteggiata
# 'linewidth=2' imposta lo spessore della linea
# 'label' imposta l'etichetta della linea nella legenda, mostrando
# il valore della media formattato con 2 cifre decimali tramite f-string

plt.title('Distribuzione delle Kill Totali nel LEC 2025', fontsize=15)
# 'plt.title()' imposta il titolo del grafico, 'fontsize=15' imposta la dimensione del testo

plt.xlabel('Numero di Kill nel Match', fontsize=12)
# 'plt.xlabel()' imposta l'etichetta dell'asse delle x, 'fontsize=12' imposta la dimensione del testo

plt.ylabel('Numero di Partite', fontsize=12)
# 'plt.ylabel()' imposta l'etichetta dell'asse delle y, 'fontsize=12' imposta la dimensione del testo

plt.legend()
# 'plt.legend()' mostra la legenda del grafico, in questo caso mostra l'etichetta
# della linea verticale creata con 'plt.axvline()'

plt.savefig('figures/distribuzione_kills_totali.png', dpi=150, bbox_inches='tight')
plt.show()
plt.savefig('figures/distribuzione_kills_totali_1.png', dpi=150, bbox_inches='tight')
# 'plt.show()' renderizza e visualizza tutti i grafici costruiti
# ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

La distribuzione delle kill totali per partita presenta un'asimmetria positiva, con una coda destra prolungata determinata dalla presenza di alcune partite caratterizzate da un numero particolarmente elevato di uccisioni. Tali osservazioni contribuiscono a innalzare il valore della media, che si attesta a **27,11 kill per partita**. La maggior parte delle osservazioni si concentra tuttavia nell'intervallo compreso approssimativamente tra 17 e 32 kill, indicando che la maggioranza delle partite presenta un numero di uccisioni inferiore al valore medio complessivo.

## 1.6 Impatto del *first blood* sull'esito finale

**Quanto incide mediamente la *first blood* sull'esito finale di una partita?**

In [ ]:
firstblood_df = (                                        # Inizializza il nuovo DataFrame 'firstblood_df'
    
    lec_df                                               # DataFrame di partenza
    
    .loc[lec_df['position'] == 'team',                   # '.loc' filtra le righe mantenendo solo quelle con 'position == team'
         ['gameid', 'side', 'result', 'firstblood']]     # e seleziona solo le colonne rilevanti per l'analisi
    
    .copy()                                              # Crea una copia indipendente evitando che 
                                                         # modifiche a 'firstblood_df' influenzino 'lec_df'
)

# Controllo qualità: verifica che ogni partita abbia esattamente 1 firstblood
firstblood_df.groupby('gameid')['firstblood'].sum().value_counts()
# '.groupby('gameid')' raggruppa i dati per gameid
# '['firstblood']' seleziona la colonna firstblood
# '.sum()' somma i valori di firstblood per ogni gameid (dovrebbe essere sempre 0+1=1)
# '.value_counts()' conta quante volte appare ogni risultato della somma

In [ ]:
fb_summary = (                        # Crea il DataFrame 'fb_summary'
    
    firstblood_df                     # DataFrame di partenza
    
    .groupby('firstblood')            # Raggruppa i dati per i valori della colonna 'firstblood'
    
    .agg(                             # .agg() applica una o più operazioni ai dati raggruppati dal .groupby() precedente
                                      # In questo caso si utilizza 
                                      # la sintassi "Named Aggregation": nome_colonna = ('colonna_sorgente', 'funzione')
        
        partite=('gameid', 'count'),  # Crea la colonna 'partite' contando le righe per ogni gruppo
        
        vittorie=('result', 'sum'),   # Crea la colonna 'vittorie' sommando i valori della colonna 'result'
        
        win_rate=('result', 'mean')   # Crea la colonna 'win_rate' calcolando la media della colonna 'result'
        
    )                                 
    .reset_index()                    # .reset_index() normalizza il DataFrame riportando 'firstblood'
                                      # come colonna normale e reimpostando l'indice con numeri progressivi
)

fb_summary['firstblood_label'] = fb_summary['firstblood'].map({
    0: 'No First Blood',
    1: 'First Blood'
})
# Crea la colonna 'firstblood_label' mappando i valori booleani della colonna 'firstblood'
# ad etichette testuali tramite un dizionario: 0 → 'No First Blood', 1 → 'First Blood'

fb_summary.head()  # Mostra le prime 5 righe del DataFrame 'fb_summary'
                   # in questo caso il DataFrame ha solo 2 righe quindi le mostra entrambe

In [ ]:
plt.figure(figsize=(6, 4))         # Inizializza l'oggetto 'figure' impostando la dimensione della tavola 
                                   # su cui verrà disegnato il grafico

ax = sns.barplot(                  # Crea e inizializza l'oggetto 'Axes' salvandolo in ax, con la creazione di un grafico a barre 
                                   # tramite Seaborn
    
    data=fb_summary,               # Parametro 'data': DataFrame sorgente del grafico
    
    x='firstblood_label',          # Parametro 'x': i valori della colonna 'firstblood_label' verranno visualizzati sull'asse delle x
    
    y='win_rate'                   # Parametro 'y': i valori della colonna 'win_rate' verranno visualizzati sull'asse delle y
)
ax.set_ylabel('Win rate')          # ax.set_ylabel() imposta l'etichetta testuale dell'asse delle y

ax.set_title('Impatto del First Blood sul risultato – LEC 2025')  # ax.set_title() imposta il titolo del grafico

ax.set_ylim(0, 1)                  # ax.set_ylim() imposta i valori minimo e massimo della scala sull'asse delle y

for c in ax.containers:            # ax.containers è una tupla di oggetti BarContainer
                                   # il ciclo for itera su ogni container della tupla
    
    ax.bar_label(                  # Aggiunge le etichette testuali sopra le barre del container corrente
        
        c,                         # Il container correntemente in elaborazione dal ciclo for
        
        labels=[f'{v*100:.1f}%' for v in c.datavalues],   # f-string che converte i valori decimali in percentuale
                                                          # con 1 cifra decimale, prendendoli da 'c.datavalues'
        
        padding=3                  # Distanza in punti tra la fine della barra e l'etichetta
    )
plt.savefig('figures/impatto_firstblood_sul_risultato.png', dpi=150, bbox_inches='tight')
plt.show()                         # Renderizza e visualizza tutti i grafici costruiti
                                   # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

L'analisi mostra che i team che conquistano la *first blood* presentano una probabilità di vittoria superiore rispetto a quelli che non la ottengono. Più precisamente, il win rate passa da circa **45,4%** per i team senza first blood a circa **54,6%** per quelli che la conquistano, con un differenziale di circa 9 punti percentuali. Tale risultato suggerisce un'associazione positiva tra *early aggression* ed esito finale, pur non implicando un rapporto di causalità diretta.

## 1.7 Partite *snowball*

**Qual è la frequenza di partite *snowball* — definite come partite con un vantaggio di oltre X gold a 15 minuti — nel LEC 2025?**

In [ ]:
snowball_df = (                                          # Nome del DataFrame che si sta creando
    
    lec_df                                               # DataFrame di origine
    
    .loc[lec_df['position'] == 'team',                   # '.loc' filtra il DataFrame mantenendo le righe
         ['gameid', 'golddiffat15']]                     # che hanno 'team' come valore nella colonna 'position'
                                                         # e seleziona solo le colonne 'gameid' e 'golddiffat15'
    
    .drop_duplicates('gameid')                           # Elimina i valori duplicati della colonna 'gameid'
                                                         # in modo da avere una sola riga per partita
    
    .copy()                                              # Crea una copia indipendente del risultato, evitando che
                                                         # modifiche a 'snowball_df' influenzino 'lec_df'
)

In [ ]:
sb_treshold = 3000                                     # Variabile che imposta la soglia minima di differenza oro
                                                       # al minuto 15 per considerare una partita come 'snowball'

snowball_df['snowball'] = (                            # Crea la colonna 'snowball' nel DataFrame 'snowball_df'
    
    snowball_df['golddiffat15'].abs() > sb_treshold    # La colonna assume valore True se il valore assoluto
                                                       # di 'golddiffat15' supera la soglia di 3000, False altrimenti
)
labels = ['Snowball', 'Non snowball']                  # Crea la variabile 'labels' composta da una lista di stringhe
                                                       # che rappresentano le etichette delle categorie

counts = [                                             # Crea una lista con due valori numerici
    
    snowball_df['snowball'].sum(),                     # Il primo valore è il conteggio delle partite 
                                                       # 'snowball' (True = 1), ottenuto sommando 
                                                       # i valori booleani della colonna 'snowball'
    
    (~snowball_df['snowball']).sum()                   # Il secondo valore è il conteggio delle partite
                                                       # 'non snowball', ottenuto invertendo la colonna
                                                       # booleana con '~' e sommando i valori risultanti
]

total = sum(counts)                                    # Calcola il totale delle partite sommando i due valori della lista 'counts'

percentages = [c / total * 100 for c in counts]        # Crea una lista con le percentuali di ogni categoria,
                                                       # dividendo ogni valore 'c' di 'counts' per il totale 
                                                       # e moltiplicando per 100 tramite list comprehension

colors = ['#1f77b4', '#ff7f0e']                        # Lista di stringhe contenenti i codici esadecimali 
                                                       # dei colori da assegnare alle fette del grafico

# Creazione Donut chart
fig, ax = plt.subplots(figsize=(6, 6))                 # Inizializza esplicitamente gli oggetti 'Figure' e 'Axes' 
                                                       # tramite 'plt.subplots()', assegnandoli alle variabili 'fig' e 'ax'
                                                       # 'figsize' imposta la dimensione della tavola in pollici

wedges, _ = ax.pie(                                    # 'ax.pie()' crea un grafico a torta e restituisce due oggetti:
                                                       # 'wedges' contiene le fette del grafico (usate dopo per la legenda)
                                                       # '_' contiene le etichette delle fette, non utilizzate in questo caso
                                                       # poiché 'labels=None', quindi scartate tramite la convenzione '_'   
    
    counts,                                            # Parametro 'x' della documentazione: dati utilizzati per costruire il grafico
    
    labels=None,                                       # Parametro che regola le etichette degli spicchi,
                                                       # 'None' indica che non devono esserci etichette sulle fette
    
    colors=colors,                                     # Parametro che regola i colori degli spicchi
    
    startangle=90,                                     # Parametro che indica l'angolo da cui il grafico 
                                                       # inizia ad essere disegnato, in questo caso 90°
    
    wedgeprops=dict(width=0.4, edgecolor='white')      # Parametro che tramite un dizionario regola le proprietà degli spicchi:
                                                       # 'width=0.4' imposta lo spessore degli spicchi creando l'effetto "donut"
                                                       # 'edgecolor=white' imposta il colore del bordo degli spicchi
)

# Legenda custom con percentuali
legend_labels = [                                      # Crea una lista di stringhe per la legenda
    f'{labels[i]} — {percentages[i]:.1f}%'             # ogni stringa combina l'etichetta e la percentuale
    for i in range(len(labels))                        # 'range(len(labels))' genera gli indici (0, 1) per
                                                       # accedere all'i-esimo elemento di 'labels' e 'percentages'
]

ax.legend(                                                   # Funzione che aggiunge la legenda al grafco
    
    wedges,                                                  # Parametro handles della documentazione ufficiale, 
                                                             # lista di elementi grafici da aggiungere al grafico, 
                                                             # si utilizza insieme a "labels" per avere il controllo 
                                                             # sulla parte descrittiva del grafico
    
    legend_labels,                                           # Parametro labels della documentazione ufficiale,       
                                                             # lista di etichette testuali che vengono renderizzate vicino 
                                                             # gli elementi grafici, si utilizza insieme al parametro "wedges" 
                                                             # per avere il controllo sulla parte descrittiva del grafico
    
    title='Tipo di partita',                                 # Imposta il titolo della legenda
    
    loc='center',                                            # Parametro che regola la posizione della legenda,
                                                             # in questo caso impostata al centro
    
    bbox_to_anchor=(0.5, 0.5),                               # Parametro che definisce le coordinate (x, y) per il 
                                                             # posizionamento preciso della legenda, in questo caso 
                                                             # (0.5, 0.5) la posiziona esattamente al centro del grafico.
                                                             # Lavora insieme a "loc='center" che definisce quale punto 
                                                             # della legenda usare come riferimento per il posizionamento
    
    frameon=False                                            # Rimuove il bordo/riquadro attorno alla legenda,
                                                             # in questo caso necessario perché la legenda è 
                                                             # posizionata al centro del donut chart
)

ax.set_title('Frequenza di partite snowball – LEC 2025')     # ax.set_title() imposta il titolo del grafico


plt.savefig('figures/frequenza_partite_snowball.png', dpi=150, bbox_inches='tight')
plt.show()                                                   # plt.show() renderizza e visualizza tutti i grafici costruiti
                                                             # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

Adottando come soglia operativa per la definizione di partita *snowball* un vantaggio in gold superiore alle 3.000 unità entro i primi 15 minuti, risulta che circa il **15%** delle partite del LEC 2025 ricade in tale categoria. Il dato suggerisce che, nella maggioranza dei match, il gioco rimanga relativamente equilibrato nelle fasi iniziali, con una quota contenuta di partite caratterizzate da un forte vantaggio economico precoce di una delle due squadre.

## 1.8 Impatto del primo Baron Nashor

**Ottenere il primo Baron aumenta significativamente la probabilità di vittoria?**

In [ ]:
baron_df = (
    lec_df
    .loc[lec_df['position'] == 'team',
         ['gameid', 'result', 'firstbaron']]
    .copy()
)
baron_df.head()

In [ ]:
baron_summary = (
    baron_df
    .groupby('firstbaron')
    .agg(
        osservazioni=('gameid', 'count'),
        vittorie=('result', 'sum'),
        win_rate=('result', 'mean')
    )
    .reset_index()
)

baron_summary['firstbaron_label'] = baron_summary['firstbaron'].map({
    False: 'No First Baron',
    True: 'First Baron'
})
baron_summary.head()

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=baron_summary,
    x='firstbaron_label',
    y='win_rate'
)

ax.set_ylabel('Win rate')
ax.set_title('Impatto del First Baron sul risultato – LEC 2025')
ax.set_ylim(0, 1)

for c in ax.containers:
    ax.bar_label(
        c,
        labels=[f'{v*100:.1f}%' for v in c.datavalues],
        padding=3
    )

plt.savefig('figures/impatto_firstbaron_sul_risultato.png', dpi=150, bbox_inches='tight')
plt.show()

### Osservazione grafico

Dal grafico si osserva che i team che ottengono il primo Baron Nashor presentano un win rate particolarmente elevato (circa **84,5%**), in netto contrasto con i team che non lo conquistano (circa **21,5%**). Una prima lettura del dato potrebbe suggerire che il primo Baron incrementi in modo marcato la probabilità di vittoria.

Tuttavia, confrontando questo risultato con l'analisi del vantaggio in gold a 15 minuti emerge una dinamica analoga: anche in quel caso, circa l'84% delle partite in cui una squadra detiene un vantaggio economico rilevante si conclude con una vittoria. Considerando inoltre che la maggior parte delle partite ha durata compresa tra 27 e 32 minuti e che il Baron Nashor diventa disponibile a partire dal 25° minuto, è plausibile interpretare il primo Baron non come causa primaria della vittoria, bensì come **strumento di consolidamento e finalizzazione** di un vantaggio già acquisito.

## 1.9 Partite chiuse senza obiettivi maggiori

**Quante partite vengono chiuse senza Baron Nashor né *Dragon Soul*?**

In [ ]:
objectives_df = (
    lec_df
    .loc[lec_df['position'] == 'team',
         ['gameid', 'dragons', 'firstbaron']]
    .copy()
)
objectives_df['dragonsoul'] = objectives_df['dragons'] >= 4
objectives_df['major_objective'] = (
    objectives_df['firstbaron'] | objectives_df['dragonsoul']
)
objectives_df.head()

In [ ]:
game_objectives = (                             # Crea il DataFrame "game_objectives"
    
    objectives_df                               # DataFrame di partenza
    
    .groupby('gameid')['major_objective']       # Raggruppa i dati per 'gameid' e 
                                                # seleziona la colonna 'major_objective' 
                                                # su cui effettuare le operazioni successive
    
    .any()                                      # '.any()' è un metodo pandas che per ogni gruppo restituisce True 
                                                # se almeno un valore è True, False se tutti i valori sono False.
                                                # In questo caso controlla i valori di 'major_objective' per ogni 
                                                # 'gameid', restituendo True se almeno uno dei due team (Blue o Red) 
                                                # ha conquistato un obiettivo maggiore

    .reset_index(name='any_major_objective')    # '.reset_index()' normalizza il DataFrame riportando 'gameid' come colonna normale
                                                # Il parametro 'name' rinomina la colonna su cui sono state applicate le operazioni
                                                # (in questo caso 'major_objective') in 'any_major_objective'
)
game_objectives['no_major_objectives'] = (      # Crea la colonna 'no_major_objectives' invertendo i valori booleani 
 ~game_objectives['any_major_objective']        # della colonna 'any_major_objective' tramite l'operatore '~'
                                                # True diventa False e viceversa, identificando le partite 
                                                # senza alcun obiettivo maggiore conquistato
)

game_objectives.head(10)                        # Mostra le prime 10 righe del dataframe

In [ ]:
# -------------------------
# DATI
# -------------------------
labels = ['Senza Baron o Dragon Soul', 'Con Baron o Dragon Soul']      # Crea una lista di stringhe con le due categorie del grafico

counts = [                                                             # Crea una lista con due valori numerici:
    
    game_objectives['no_major_objectives'].sum(),                      # [0] Totale partite SENZA obiettivi maggiori,
                                                                       # ottenuto sommando i True di 'no_major_objectives'
    
    (~game_objectives['no_major_objectives']).sum()                    # [1] Totale partite CON almeno un obiettivo maggiore,
                                                                       # ottenuto invertendo i valori booleani la colonna con '~' e sommando
]

total = sum(counts)                                                    # Somma entrambi i valori di counts ottenendo il numero totale di partite

pct1 = counts[0] / total * 100                                         # Calcola la percentuale di partite senza obiettivi maggiori
                                                                       # dividendo il primo valore di 'counts' per il totale e moltiplicando per 100

pct2 = counts[1] / total * 100                                         # Calcola la percentuale di partite con almeno un obiettivo maggiore
                                                                       # dividendo il secondo valore di 'counts' per il totale e moltiplicando per 100

# -------------------------
# COLORI
# -------------------------
violet = '#7b5fc3'                          # Variabile che contiene il codice esadecimale del colore viola
grey = '#bdbdbd'                            # Variabile che contiene il codice esadecimale del colore grigio
colors = [violet, grey]                     # Lista che raccoglie i colori da assegnare alle fette del grafico

# -------------------------
# DONUT
# -------------------------
fig, ax = plt.subplots(figsize=(6, 6))

ax.pie(                                            # Crea il grafico a torta, il valore di ritorno ('wedges, _') non viene salvato
                                                   # perché la legenda viene gestita manualmente tramite ax.text() e ax.add_patch()
    
    counts,                                        # Parametro 'x' nella documentazione ufficiale: dati con cui viene costruito il grafico,
                                                   # in questo caso i valori della lista 'counts'
    
    labels=None,                                   # Parametro che regola le etichette sugli spicchi,
                                                   # 'None' indica di non mostrare nessuna etichetta
    
    colors=colors,                                 # Parametro che regola i colori degli spicchi,
                                                   # in questo caso i colori della lista 'colors'
    
    startangle=90,                                 # Parametro che definisce l'angolo da cui inizia
                                                   # la renderizzazione del grafico, in questo caso 90°
    
    wedgeprops=dict(                               # Parametro che tramite un dizionario regola le proprietà degli spicchi:
        width=0.4,                                 # 'width' imposta lo spessore degli spicchi creando l'effetto donut
        edgecolor='white'                          # 'edgecolor' imposta il colore del bordo degli spicchi
    )  
)

ax.set_title(                                                    # Imposta il titolo del grafico
    
    'Partite concluse senza Baron o Dragon Soul – LEC 2025',     # Stringa di testo del titolo 
    
    pad=20                                                       # 'pad' regola la distanza in punti
                                                                 # tra il titolo e la parte superiore del grafico
)

# -------------------------
# "LEGENDA" CUSTOM CENTRATA
# -------------------------
 
x0, y0 = 0.5, 0.5                           # Crea le variabili 'x0' e 'y0' contenenti le coordinate centrali
                                            # del grafico in sistema 'ax.transAxes' dove (0,0)=basso-sinistra 
                                            # e (1,1)=alto-destra, quindi (0.5, 0.5) rappresenta il centro esatto

# Layout
line_h = 0.065                              # Variabile che definisce la distanza verticale tra le righe della legenda

box_w, box_h = 0.04, 0.025                  # Variabili che definiscono rispettivamente larghezza e altezza 
                                            # dei rettangoli colorati della legenda

x_shift = 0.02                              # Variabile che definisce la separazione orizzontale 
                                            # tra la percentuale e il rettangolo colorato

# Titolo legenda
ax.text(                                    # Aggiunge del testo all'oggetto 'Axes'
    
    x0, y0 + 2.1*line_h,                    # Parametri 'x' e 'y': coordinate di posizionamento del testo,
                                            # 'y' è ottenuto sommando 'y0' a '2.1*line_h' per posizionare
                                            # il titolo sopra gli altri elementi della legenda
    
    "Tipo di partita",                      # Parametro 's': stringa di testo che verrà renderizzata
    
    ha='center', va='center',               # 'ha' e 'va': allineamento orizzontale e verticale
                                            # del testo relativo al punto di ancoraggio
    
    fontsize=13,                            # Regola la dimensione del testo
    
    fontweight='bold',                      # Regola lo spessore del testo, 'bold' lo rende in grassetto
    
    transform=ax.transAxes                  # Imposta il sistema di coordinate come 'transAxes',
                                            # dove (0,0)=basso-sinistra e (1,1)=alto-destra del grafico.
                                            # Senza questo parametro le coordinate verrebbero interpretate
                                            # come valori dei dati invece che come posizioni relative al grafico
)

# --- BLOCCO 1 (Snowball/No Baron+Soul) ---
# percentuale (a sinistra del centro)
ax.text(                                    # Aggiunge del testo all'oggetto Axes
    
    x0 - x_shift, y0 + 0.8*line_h,          # Parametri 'x' e 'y': coordinate di posizionamento del testo,
                                            # 'x' è spostato a sinistra del centro tramite 'x_shift'
                                            # 'y' è posizionato sopra il centro tramite '0.8*line_h'
    
    f"{pct1:.1f}%",                         # Parametro 's': f-string che renderizza la percentuale
                                            # della prima categoria con 1 cifra decimale e simbolo '%'

    ha='right', va='center',                # 'ha' e 'va': allineamento orizzontale a destra e
                                            # verticale al centro del punto di ancoraggio

    fontsize=12,                            # Regola la dimensione del testo

    transform=ax.transAxes                  # Imposta il sistema di coordinate come 'transAxes'
)

# rettangolo (a destra del centro)
ax.add_patch(mpatches.Rectangle(                    # 'ax.add_patch()' aggiunge una forma geometrica all'Axes
                                                    # 'mpatches.Rectangle()' crea un rettangolo colorato
    
    (x0 + x_shift, y0 + 0.8*line_h - box_h/2),      # Coordinate (x, y) dell'angolo in basso a sinistra
                                                    # del rettangolo, posizionato a destra della percentuale
    
    box_w, box_h,                                   # Larghezza e altezza del rettangolo
    
    transform=ax.transAxes,                         # Sistema di coordinate relativo al grafico
    
    facecolor=violet,                               # Colore di riempimento del rettangolo
    
    edgecolor='white'                               # Colore del bordo del rettangolo
))

# descrizione (riga sotto)
ax.text(                                            # Aggiunge del testo all'oggetto 'Axes' 
    
    x0, y0 + 0.05*line_h,                           # Parametri 'x' e 'y': coordinate di posizionamento del testo,
                                                    # 'x' assume il valore della variabile 'x0'
                                                    # 'y' è posizionato sopra il centro tramite '0.05*line_h'
    
    labels[0],                                      # Parametro 's': testo da renderizzare, in questo caso è il primo valore della variabile 'labels'
    
    ha='center', va='center',                       # 'ha' e 'va': allineamenti orizzontale e verticale, entrambi al centro
    
    fontsize=12,                                    # Regola la dimensione del testo
    
    transform=ax.transAxes                          # Imposta il sistema di coordinate come 'transAxes'                  
)

# --- BLOCCO 2 (Con Baron o Dragon Soul) ---
ax.text(                                            # Aggiunge del testo all'oggetto 'Axes'
    
    x0 - x_shift, y0 - 1.05*line_h,                 # Parametri 'x' e 'y': coordinate di posizionamento del testo,
                                                    # 'x' è spostato a sinistra del centro tramite 'x_shift'
                                                    # 'y' è posizionato sotto il centro tramite '1.05*line_h'     
    
    f"{pct2:.1f}%",                                 # Parametro 's': f-string che renderizza la percentuale
                                                    # della seconda categoria con 1 cifra decimale e simbolo '%

    ha='right', va='center',                        # 'ha' e 'va': allineamento orizzontale a destra e
                                                    # verticale al centro del punto di ancoraggio
 
    fontsize=12,                                    # Regola la dimensione del testo

    transform=ax.transAxes                          # Imposta il sistema di coordinate come 'transAxes'
)

ax.add_patch(mpatches.Rectangle(                    # 'ax.add_patch()' aggiunge una forma geometrica all'Axes
                                                    # 'mpatches.Rectangle()' crea un rettangolo colorato
    
    (x0 + x_shift, y0 - 1.05*line_h - box_h/2),     # Coordinate (x, y) dell'angolo in basso a sinistra
                                                    # del rettangolo, posizionato a destra della percentuale
    
    box_w, box_h,                                   # Larghezza e altezza del rettangolo
    
    transform=ax.transAxes,                         # Sistema di coordinate relativo al grafico
    
    facecolor=grey,                                 # Colore di riempimento del rettangolo, in questo caso grigio
    
    edgecolor='white'                               # Colore del bordo del rettangolo
))

ax.text(                                            
    x0, y0 - 1.8*line_h,                            # Parametri 'x' e 'y': coordinate di posizionamento del testo,
                                                    # 'x' assume il valore della variabile 'x0'
                                                    # 'y' è posizionato sotto il centro tramite '1.8*line_h'
    labels[1],                                      # Parametro 's': testo da renderizzare, 
                                                    # in questo caso il secondo valore della variabile 'labels'
    ha='center', va='center',                       # 'ha' e 'va': allineamenti orizzontale e verticale, entrambi al centro
    fontsize=12,                                    # Regola la dimensione del testo
    transform=ax.transAxes                          # Imposta il sistema di coordinate come 'transAxes'
)

plt.savefig('figures/partite_con_baron_o_dragonsoul.png', dpi=150, bbox_inches='tight')
plt.show()                                          # Renderizza e visualizza tutti i grafici costruiti
                                                    # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

Il **92%** delle partite del LEC 2025 si conclude con la conquista di almeno uno tra il primo Baron e la *Dragon Soul*, evidenziando la centralità degli obiettivi maggiori nella risoluzione dei match. Solo una quota marginale di partite si conclude in assenza di entrambi gli obiettivi, suggerendo che quasi tutti gli incontri raggiungano la fase di gioco nella quale tali obiettivi diventano contendibili.

# 2. Performance dei team

La presente sezione si concentra sul confronto fra le squadre partecipanti al LEC 2025, esaminando indicatori sintetici di vantaggio economico, controllo degli obiettivi neutrali, stile di gioco e capacità di gestire vantaggi e svantaggi iniziali.

## 2.1 Gold difference medio a 15 minuti per team

**Quali team del LEC hanno il *gold difference* medio più elevato a 15 minuti?**

In [ ]:
avg_goldAt15_team = (                     # Creazione del DataFrame 'avg_goldAt15_team'
    
    lec_df                                # DataFrame di origine
    
    .loc[lec_df['position'] == 'team',    # '.loc' con il primo argomento filtra tutte le righe del DataFrame che hanno 'team' come valore della
         ['teamname', 'golddiffat15']]    # colonna position, mentre con il secondo argomento gestisce le colonne del DataFrame di origine da mantenere
                                          # in quello finale
    
    .copy()                               # Crea una copia indipendente del DataFrame creato 
)

avg_goldAt15_team.head(20)                # Mostra le prime 20 righe del DataFrame

In [ ]:
avg_gold_team = (                                  # Crea il DataFrame 'avg_gold_team'
    
    avg_goldAt15_team                              # DatFrame di origine
    
    .groupby('teamname', as_index=False)           # Raggruppa i dati per i valori di 'teamname',
                                                   # 'as_index=False' mantiene 'teamname' come colonna normale evitando che
                                                   # diventi l'indice (è un metodo alternativo a '.reset_index()')
    
    .agg(                                          # Funzione di aggregazione, serve per eseguire operazioni specifiche sui dati
        
        avg_golddiff15=('golddiffat15', 'mean')    # Crea la colonna 'avg_golddiff15' calcolando la media della colonna 'golddiffat15'
    )
)

avg_gold_team = avg_gold_team.sort_values(    # '.sort_values()' ordina le righe del DataFrame in base ai valori di una colonna specificata
    'avg_golddiff15',                         # in questo caso si ordina sui valori della colonna 'avg_golddiff15'
    ascending=False                           # 'ascending=False' imposta l'ordine decrescente
                                              # (dal valore più alto al più basso)
)

avg_gold_team.head(20)                        # Mostra le prime 20 righe del DataFrame

In [ ]:
plt.figure(figsize=(15, 7))                                     # Inizializza l'oggetto 'Figure', cioè la tela su cui verrà renderizzato il grafico
                                                                # ed imposta la sua grandezza tramite 'figsize'

ax = sns.barplot(                                               # Crea l'oggetto 'Axes' tramite la creazione di un grafico a barre con Seaborn
    
    data=avg_gold_team,                                         # Il parametro 'data' indica il DataFrame che verrà utilizzato per la creazione
                                                                # del grafico. In questo caso verrà utilizzato il DataFrame 'avg_gold_team'
    
    x='avg_golddiff15',                                         # Il parametro 'x' indica quale colonna del DataFrame verrà utilizzata sull'asse delle x.
                                                                # In questo caso verranno utilizzati i dati della colonna 'avg_golddiff15'
    
    y='teamname',                                               # Il parametro 'y' indica quale colonna del Dataframe verrà utilizzata sull'asse delle y.
                                                                # In questo caso verranno utilizzati i dati della colonna 'teamname'
    
    color='#4c72b0'                                             # Il parametro 'color' regola il colore delle barre del grafico
) 

ax.set_title(                                                   # Imposta il titolo del grafico
    
    'Gold Difference medio a 15 minuti per Team – LEC 2025',    # Stringa di testo che sarà effetticamente il testo renderizzato
    
    pad=15                                                      # Il parametro 'pad' regola la distanza in punti 
                                                                # tra il titolo e la parte superiore del grafico
)
ax.set_xlabel('Gold Difference medio a 15 minuti')              # Imposta l'etichetta dell'asse delle x

ax.set_ylabel('Team')                                           # Imposta l'etichetta dell'asse delle y

# linea di riferimento a 0
ax.axvline(                          # Crea una linea verticale nel grafico
    
    0,                               # Parametro 'x': posiziona la linea a 0,
                                     # serve come riferimento per distinguere i team
                                     # con gold difference positiva da quelli con negativa
    
    color='black',                   # Parametro che regola il colore della linea
    
    linewidth=1                      # Parametro che regola lo spessore della linea in punti
)                   

# etichette valori
for c in ax.containers:              # 'ax.containers' è una tupla di oggetti BarContainer
                                     # il ciclo for itera su ogni container della tupla
    
    ax.bar_label(                    # Aggiunge le etichette testuali sopra le barre 
                                     # del container correntemente in elaborazione
        
        c,                           # Il container correntemente in elaborazione dal ciclo for
        
        fmt='%.0f',                  # Formatta le etichette come numeri interi senza cifre decimali
                                     # ('%.0f' è la sintassi in stile C: '.0'=zero decimali, 'f'=float)
        
        padding=3                    # Distanza in punti tra la fine della barra e l'etichetta
    )
    
plt.savefig('figures/gold_diff_15min_per_team.png', dpi=150, bbox_inches='tight')
plt.show()                           # Renderizza e visualizza tutti i grafici costruiti
                                     # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

Dal grafico emerge che solo **5 team su 11** presentano una *gold difference* media positiva a 15 minuti. Tra questi, solamente 3 team mostrano un vantaggio medio chiaramente distinto dallo zero, mentre gli altri si collocano in prossimità della parità.

Tale distribuzione suggerisce che il vantaggio economico early game nel LEC 2025 sia concentrato in un numero ristretto di squadre, mentre la maggioranza dei team tende a trovarsi in situazioni di sostanziale equilibrio o di lieve svantaggio nei primi 15 minuti. La competizione sembra dunque strutturarsi attorno a pochi team in grado di imporre sistematicamente un vantaggio nelle fasi iniziali, piuttosto che presentare un equilibrio diffuso tra tutte le squadre.

## 2.2 Relazione tra gold diff @15 e win rate finale

**Esiste una relazione tra *gold difference* a 15 minuti e win rate finale del team?**

In [ ]:
team_gold_win = lec_df[                                   # Creazione del DataFrame 'team_gold_win' creato a partire dal DataFrame
    ['position', 'teamname', 'golddiffat15', 'result']    # 'lec_df' e filtrando da quest'ultimo solo alcune colonne
]

team_gold_win = team_gold_win[                            # Filtraggio delle righe che hanno come valore 'team' nella colonna position
    team_gold_win['position'] == 'team'
]

team_gold_win = (                                         # Modifica del DataFrame 'team_gold_win' tramite alcune operazioni su se stesso
    
    team_gold_win                                         # DataFrame di partenza
    
    .groupby(['teamname', 'result'], as_index=False)      # '.groupby' raggruppa i dati per 'teamname' e 'result', 'as_index=False'
                                                          # fa in modo che 'teamname' e 'result' non vengano trattati come indici ma come
                                                          # colonne normali
    
    .agg(                                                 # '.agg' è una funzione di aggregazione che permette di eseguire varie operazioni
                                                          # su colonne specifiche
        
        avg_golddiffat15=('golddiffat15', 'mean'),        # Crea la colonna 'avg_golddiffat15' eseguendo la media della colonna
                                                          # 'golddiffat15'
        
        games=('golddiffat15', 'count')                   # Crea la colonna 'games' conteggiando le entry della colonna 'golddiffat15'
    )
)

team_gold_win_pivot = team_gold_win.pivot(                    # Rimodella il DataFrame 'team_gold_win_pivot' producendo una tabella pivot
    
    index='teamname',                                         # Colonna da utilizzare per stabilire gli indici del nuovo DataFrame
    
    columns='result',                                         # Colonna da utilizzare per stabilire le colonne del nuovo DataFrame
    
    values='avg_golddiffat15'                                 # Colonna da utilizzare per aggiungere i valori al nuovo DataFrame
    
).reset_index()                                               # Riporta 'teamname' a colonna normale e reimposta gli indici come numeri
                                                              # progressivi

team_gold_win_pivot.columns.name = None                       # Rimuove il nome dell'asse delle colonne ('result')
                                                              # aggiunto automaticamente da '.pivot()',
                                                              # operazione di pulizia estetica del DataFrame

team_gold_win_pivot = team_gold_win_pivot.rename(columns={    # Rinomina le colonne 'False' e 'True' del DataFrame 'team_gold_win_pivot'
    False: 'avg_gold_loss',                                   # in rispettivamente 'avg_gold_loss' e 'avg_gold_win'
    True: 'avg_gold_win'
})
team_gold_win_pivot.head(10)                                  # Mostra le prime 10 righe del DataFrame

In [ ]:
df = team_gold_win_pivot.sort_values('avg_gold_win')     # Crea la variabile 'df' che contiene il dataframe 'team_gold_win_pivot'
                                                         # con i valori ordinati in modo ascendente tramite la funzione '.sort_values'

plt.figure(figsize=(10, 6))                              # Inizializza l'oggetto 'Figure' e imposta la sua grandezza tramite 'figsize'

plt.hlines(                                              # Crea le linee orizzontali del "Dumbbell Chart"
    
    y=df['teamname'],                                    # Parametro che regola i valori che appariranno sulla colonna delle y
    
    xmin=df['avg_gold_loss'],                            # Il punto di inizio di ciascuna linea che corrisponde al valore minimo
    
    xmax=df['avg_gold_win']                              # Il punto di fine di ciascuna linea che corrisponde al valore massimo
)

plt.scatter(                                             # Aggiunge i punti di sconfitta al Dumbbell Chart
    
    df['avg_gold_loss'],                                 # Parametro 'x': posizione orizzontale dei punti,
                                                         # corrisponde alla gold diff media in sconfitta
    
    df['teamname']                                       # Parametro 'y': posizione verticale dei punti,
                                                         # corrisponde al nome del team
)

plt.scatter(                                             # Aggiunge i punti di vittoria al Dumbbell Chart
    
    df['avg_gold_win'],                                  # Parametro 'x': posizione orizzontale dei punti,
                                                         # corrisponde alla gold diff media in vittoria
    
    df['teamname']                                       # Parametro 'y': posizione verticale dei punti,
                                                         # corrisponde al nome del team
)

plt.axvline(0)                                           # Aggiunge una linea verticale nel grafico sul punto 0 dell'asse delle x

plt.xlabel('Gold Difference medio a 15 minuti')          # Etichetta testuale dell'asse delle x

plt.title('Gold diff @15: confronto Vittorie vs Sconfitte per Team – LEC 2025')  # Variabile che imposta il titolo del grafico

plt.tight_layout()      # Aggiusta automaticamente i margini della figura per evitare
                        # che gli elementi del grafico si sovrappongano o vengano tagliati

plt.savefig('figures/gold_diff_15min_vittorie_vs_sconfitte.png', dpi=150, bbox_inches='tight')
plt.show()              # Renderizza tutti i grafici non renderizzati fino a questo punto

### Osservazione grafico

Il *dumbbell chart* rappresenta la relazione tra il *gold difference* medio a 15 minuti e l'esito della partita, distinguendo tra vittorie e sconfitte, per ciascun team. Complessivamente si osserva che tutti i team presentano un vantaggio economico early game nettamente positivo nelle partite vinte, mentre mostrano valori negativi o prossimi allo zero nelle sconfitte, evidenziando una forte associazione tra *early advantage* ed esito finale.

Emergono tuttavia due casi particolari. **Natus Vincere** non presenta osservazioni di vittoria; il confronto risulta pertanto limitato alle sole sconfitte, associate a un marcato svantaggio economico già nei primi 15 minuti. **G2 Esports** rappresenta invece un'eccezione rilevante: è l'unico team che mantiene in media un *gold difference* positivo a 15 minuti anche nelle partite perse, suggerendo una capacità di generare vantaggio economico early game indipendentemente dall'esito finale del match.

## 2.3 Conversione del vantaggio iniziale in vittoria

**Quali team convertono meglio il vantaggio iniziale in vittoria?**

In [ ]:
conversion_df = lec_df[                                    # Creazione del DataFrame 'conversion_df' partendo dal DataFrame 'lec_df'
    ['position', 'teamname', 'golddiffat15', 'result']     # mantenendo solo le colonne ['position', 'teamname', 'golddiffat15', 'result']
]

conversion_df = conversion_df[                             # Si filtrano le righe del DataFrame mantenendo solo quelle che hanno 'team'
    conversion_df['position'] == 'team'                    # come valore di 'position'
]
conversion_df['early_advantage'] = conversion_df['golddiffat15'] > 0   # Crea la colonna 'early_advantage' che assume 
                                                                       # True se 'golddiffat15' > 0
                                                                       # (il team aveva vantaggio oro a 15 minuti), False altrimenti

conversion_df = conversion_df[                                         # Filtra il DataFrame mantenendo 
                                                                       # solo le righe dove 'early_advantage' è True,                                                           
    conversion_df['early_advantage']                                   # ovvero solo le partite in cui il team 
                                                                       # aveva vantaggio oro a 15 minuti
]

team_conversion = (                                                    # Crea il DataFrame 'team_conversion'
    
    conversion_df                                                      # DataFrame di partenza
    
    .groupby('teamname')                                               # Raggruppa di dati per i valori della colonna 'teamname'
    
    .agg(                                                              # La funzione '.agg' permette di eseguire alcune operazioni sui dati
        
        games_with_adv=('result', 'count'),                            # Crea la colonna 'games_with_adv' contando i valori 
                                                                       # della colonna 'result'
        
        wins_with_adv=('result', 'sum')                                # Crea la colonna 'wins_with_adv' sommando i valori
                                                                       # della colonna 'result'
    )
    .reset_index()                                                     # Reset index riporta la colonna 'teamname' da index a colonna normale
)

team_conversion['conversion_rate'] = (                                 # Crea la colonna 'conversion_rate' i cui valori sono dati 
    team_conversion['wins_with_adv'] /                                 # dal rapporto tra le vittorie con vantaggio (wins_with_adv)
    team_conversion['games_with_adv']                                  # e le partite con vantaggio (games_with_adv)
)

# Ordiniamo i team per conversion rate
plot_df = team_conversion.sort_values(                                 # Crea il DataFrame 'plot_df' composto dai dati del
    'conversion_rate',                                                 # DataFrame 'team_conversion' ordinati in modo discendente
    ascending=False
)

plt.figure(figsize=(10, 6))              # Inizializza l'oggeto 'Figure' e stabilisce la sua grandezza con 'figsize'

plt.barh(                                # Crea un grafico a barre orizzontali
    plot_df['teamname'],                 # Parametro y nella documentazione ufficiale, sono i dati che andranno sull'asse delle y
    plot_df['conversion_rate']           # Parametro 'width' nellla documentazione ufficiale, sono i dati che andranno sull'asse delle x
)

plt.xlabel('Conversion rate (win | gold diff @15 > 0)')     # Crea un'etichetta testuale sull'asse delle x
plt.title('Capacità di conversione del vantaggio early in vittoria – LEC 2025')   # Crea il titolo del grafico

# Linea di riferimento al 50%
plt.axvline(0.5, linestyle='--')  # Aggiunge una linea verticale di riferimento sul punto 0.5,
                                  # rappresenta il 50% di conversion rate come soglia di riferimento
                                  # 'linestyle='--'' imposta la linea come tratteggiata

# Annotazioni con numero di partite
for i, row in plot_df.iterrows():   # È un metodo pandas che itera su un DataFrame riga per riga, 
                                    # restituendo ad ogni iterazione due valori: 
                                    #   i: l'indice della riga
                                    #   row: la riga stessa come Series, quindi puoi accedere ai valori con row['colonna']
    
    plt.text(                            # Aggiunge del testo al grafico
        
        row['conversion_rate'] + 0.01,   # Posizione x del testo, leggermente a destra della barra
        
        row['teamname'],                 # Posizione y del testo, in corrispondenza del team
        
        f"{int(row['games_with_adv'])}", # Testo da mostrare, il numero di partite con vantaggio convertito in intero
        
        va='center'                      # Allineamento verticale al centro
    )

plt.xlim(0, 1)        # Imposta i limiti dell'asse delle x tra 0 e 1,
                      # coerente con i valori di conversion_rate che sono percentuali decimali

plt.tight_layout()    # Aggiusta automaticamente i margini della figura per evitare
                      # che gli elementi del grafico si sovrappongano o vengano tagliati

plt.savefig('figures/conversion_rate_vantaggio_per_team.png', dpi=150, bbox_inches='tight')
plt.show()              # Renderizza tutti i grafici non renderizzati fino a questo punto

### Osservazione grafico

L'analisi della *conversion rate* — definita come la quota di partite vinte fra quelle in cui il team detiene un *gold difference* positivo a 15 minuti — evidenzia che **Fnatic**, **G2 Esports** e **Team Vitality** sono i team con la maggiore capacità di trasformare il vantaggio economico iniziale in vittoria. Fnatic emerge come il team più efficiente, riuscendo a convertire oltre l'80% delle partite con vantaggio early in una vittoria finale. G2 Esports e Team Vitality presentano valori di conversione anch'essi elevati, sebbene marginalmente inferiori a quelli di Fnatic.

## 2.4 Capacità di recupero dello svantaggio iniziale

**Quali team recuperano più spesso le partite partendo in svantaggio?**

In [ ]:
recovery_df = lec_df[                                     # Creazione del DataFrame 'recovery_df' partendo dal DataFrame 'lec_df'
    ['position', 'teamname', 'golddiffat15', 'result']    # mantenendo solo le colonne rilevanti per l'analisi
]
recovery_df = recovery_df[                                # Filtra le righe del DataFrame mantenendo solo quelle che hanno 'team'
    recovery_df['position'] == 'team'                     # come valore di 'position'
]
recovery_df['early_disadvantage'] = recovery_df['golddiffat15'] < 0
# Crea la colonna 'early_disadvantage' che assume True se 'golddiffat15' < 0
# (il team era in svantaggio oro a 15 minuti), False altrimenti

recovery_df = recovery_df[
    recovery_df['early_disadvantage']                     # Filtra il DataFrame mantenendo solo le righe dove 'early_disadvantage' è True,
]                                                         # ovvero solo le partite in cui il team era in svantaggio oro a 15 minuti

team_recovery = (                                         # Crea il DataFrame 'team_recovery'
    recovery_df                                           # DataFrame di partenza
    .groupby('teamname')                                  # Raggruppa i dati per i valori della colonna 'teamname'
    .agg(                                                 # '.agg' permette di eseguire alcune operazioni sui dati
        games_from_behind=('result', 'count'),            # Crea la colonna 'games_from_behind' contando i valori della colonna 'result'
        wins_from_behind=('result', 'sum')                # Crea la colonna 'wins_from_behind' sommando i valori della colonna 'result'
    )
    .reset_index()                                        # Riporta la colonna 'teamname' da indice a colonna normale
)
team_recovery['recovery_rate'] = (                        # Crea la colonna 'recovery_rate' i cui valori sono dati
    team_recovery['wins_from_behind'] /                   # dal rapporto tra le vittorie da svantaggio (wins_from_behind)
    team_recovery['games_from_behind']                    # e le partite da svantaggio (games_from_behind)
)
plot_df = team_recovery.sort_values(                      # Crea il DataFrame 'plot_df' composto dai dati del
    'recovery_rate',                                      # DataFrame 'team_recovery' ordinati in modo discendente
    ascending=False
)
plt.figure(figsize=(10, 6))                               # Inizializza l'oggetto 'Figure' e stabilisce la sua grandezza con 'figsize'
plt.barh(                                                 # Crea un grafico a barre orizzontali
    plot_df['teamname'],                                  # Parametro 'y': dati sull'asse delle y
    plot_df['recovery_rate']                              # Parametro 'width': dati sull'asse delle x
)
plt.xlabel('Recovery rate (win | gold diff @15 < 0)')     # Imposta l'etichetta testuale dell'asse delle x
plt.title('Capacità di recupero partendo in svantaggio – LEC 2025')  # Imposta il titolo del grafico
plt.axvline(0.5, linestyle='--')                          # Aggiunge una linea verticale di riferimento sul punto 0.5,
                                                          # rappresenta il 50% di recovery rate come soglia di riferimento
                                                          # 'linestyle='--'' imposta la linea come tratteggiata

for i, row in plot_df.iterrows():                         # Itera su ogni riga del DataFrame restituendo
                                                          # 'i' come indice e 'row' come riga corrente
    plt.text(                                             # Aggiunge del testo al grafico
        row['recovery_rate'] + 0.01,                      # Posizione x del testo, leggermente a destra della barra
        row['teamname'],                                  # Posizione y del testo, in corrispondenza del team
        f"{int(row['games_from_behind'])}",               # Testo da mostrare, il numero di partite da svantaggio convertito in intero
        va='center'                                       # Allineamento verticale al centro
    )
plt.xlim(0, 1)                                            # Imposta i limiti dell'asse delle x tra 0 e 1,
                                                          # coerente con i valori di recovery_rate che sono percentuali decimali
plt.tight_layout()                                        # Aggiusta automaticamente i margini della figura per evitare
                                                          # che gli elementi del grafico si sovrappongano o vengano tagliati
plt.savefig('figures/recovery_rate_svantaggio_per_team.png', dpi=150, bbox_inches='tight')
plt.show()                                                # Renderizza e visualizza tutti i grafici costruiti
                                                          # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

La capacità di recuperare una partita partendo in svantaggio a 15 minuti risulta generalmente bassa per la maggior parte dei team del LEC 2025. La quasi totalità delle squadre presenta infatti un *recovery rate* inferiore al 50%, indicando che, nella maggioranza dei casi, uno svantaggio economico early game tende a tradursi in una sconfitta finale.

Fanno eccezione **Karmine Corp** e **G2 Esports**, gli unici team capaci di ribaltare lo svantaggio iniziale in oltre il 50% delle occasioni, evidenziando una maggiore efficacia nella gestione delle fasi di mid e late game.

## 2.5 Dragon control rate per team

**Qual è il *dragon control rate* medio — ovvero la quota media di draghi totali per partita controllati dal team — di ciascun team del LEC 2025?**

In [ ]:
control_rate_dragons_df = lec_df[['gameid', 'position', 'teamname', 'dragons']].copy()
# Crea il DataFrame 'control_rate_dragon_df' partendo dal DataFrame 'lec_df' e selezionando solo le colonne
# ['gameid', 'position', 'teamname', 'dragons'] e creando una copia del risultato in modo che i due DataFrame siano indipendenti tra loro

control_rate_dragons_df = control_rate_dragons_df[control_rate_dragons_df['position'] == 'team'].copy()
# Filtra le righe del DataFrame 'control_ratedragon_df' mantenendo solo quelle che hanno 'team' come valore nella colonna 'position'

# (opzionale ma consigliato) assicurati che dragons sia numerico
control_rate_dragons_df['dragons'] = pd.to_numeric(control_rate_dragons_df['dragons'], errors='coerce')
# Converte il dtype della colonna 'dragons' in numerico tramite 'pd.to_numeric()'
# Il parametro 'errors='coerce'' fa sì che i valori non convertibili vengano
# sostituiti con 'NaN' invece di generare un errore

control_rate_dragons_df['total_dragon_game'] = (  # Crea la colonna 'total_dragon_game' 
    control_rate_dragons_df                       # partendo dal DataFrame 'control_rate_dragons_df''
    .groupby('gameid')['dragons']                 # raggruppando per 'gameid' e operando solo sulla colonna 'dragons'
    .transform('sum')                             # '.transform()' calcola la somma dei draghi per ogni 'gameid'
                                                  # e la ripete su tutte le righe del gruppo, mantenendo
                                                  # la struttura originale del DataFrame (a differenza di '.agg()'
                                                  # che restituisce una sola riga per gruppo)
)

# 3) Dragon control rate per team-partita
#    Evitiamo divisioni per zero (es. partite con 0 draghi)
control_rate_dragons_df['dragon_control_rate'] = np.where(                              # 'where' dice ci applicare una specifica operazione
                                                                                        # alla colonna 'dragon_control_rate' quando
                                                                                        # una specifica condizione viene soddisfatta
    
    control_rate_dragons_df['total_dragon_game'] > 0,                                   # riga di codice che identifica la condizione
                                                                                        # in questo caso quando il numero
                                                                                        # 'total_dragon_game' è maggiore di 0
    
    control_rate_dragons_df['dragons'] / control_rate_dragons_df['total_dragon_game'],  # esegue il rapporto tra il valore della colonna
                                                                                        # 'dragonè e la colonna 'total_dragon_game'
    
    np.nan                                                                              # Valore assegnato quando la condizione 
                                                                                        # non è soddisfatta (total_dragon_game = 0), 
                                                                                        # 'np.nan' rappresenta un valore
                                                                                        # mancante evitando divisioni per zero
)

# 4) Media del control rate per team (media delle partite)
team_dragon_control = (                                           # Crea il DataFrame 'team_dragon_control'
    control_rate_dragons_df                                       # DataFrame di partenza
    .groupby('teamname', as_index=False)                          # Si raggruppano i dati per i valori della colonna 'teamname'
                                                                  # 'as_index=False' previene che la colonna di raggruppamento
                                                                  # diventi un indice mantenendola come colonna normale
                                                                  # (alternativa a 'reset_index')
    
    .agg(                                                         # La funzione '.agg' permette di applicare operazioni specifiche ai dati
        avg_dragon_control_rate=('dragon_control_rate', 'mean'),  # crea la colonna 'avg_dragon_control_rate' i cui valori sono le medie
                                                                  # dei valori della colonna 'dragon_control rate'
        
        games=('gameid', 'nunique')                               # Crea la colonna 'games' i cui valori sono il numero valori unici
                                                                  # della colonna 'gameid'
    )
    .sort_values('avg_dragon_control_rate', ascending=False)      # Ordina il DataFame in modo discendente sulla base dei valori 
                                                                  # della colonna 'avg_dragon_control_rate'
)

display(team_dragon_control)   # Funzione di Jupyter Notebook che visualizza il DataFrame
                               # in formato tabellare leggibile in qualsiasi punto della cella

# 5) Grafico a barre
plt.figure(figsize=(10, 6))    # Inizializza l'oggetto 'Figure' cioè la 'tela' su cui verrà renderizzato il grafico
                               # 'figsize' regola la grandezza della tela

plt.barh(                                            # '.barh' crea un grafico a barre orizzontali
    team_dragon_control['teamname'],                 # Parametro 'y' della documentazione ufficiale, idica i valori che andranno ad essere
                                                     # inseriti sull'asse delle y
    
    team_dragon_control['avg_dragon_control_rate']   # Parametro 'width' della documentazione ufficiale, regola la largezza delle barre 
                                                     # sull'asse delle x
) 

plt.xlabel('Dragon control rate medio (dragons team / dragons totali game)') # '.xlabel' imposta l'etichetta delle asse delle x

plt.title('Miglior Dragon Control Rate per Team – LEC 2025') # 'title' imposta il titolo del grafico

plt.xlim(0, 1) # '.xlim' regola minimo e massimo valore che l'asse delle x può avere

# annotazione numero partite (robustezza)
for _, row in team_dragon_control.iterrows():   # Ciclo loop che itera gli oggetti 'index' e 'row' creati da '.iterrows()' 
                                                # per ogni riga del DataFrame
                                                # index che identifica il numero di indice della riga 
                                                # (associato in questo caso a '_' siccome i valori verranno omessi)
                                                # row che identifica i dati di ogni riga come Serie 
                                                # (associato in questo caso alla variabile 'row')
                                           
                                                
    plt.text(                                   # Aggiunge del testo al grafico
        
        row['avg_dragon_control_rate'] + 0.01,  # Parametro x della documentazione ufficiale
                                                # indica la referenza per posizionare il testo sull'asse delle x
        
        row['teamname'],                        # Parametro y della documentazione ufficiale
                                                # indica la referenza per posizionare il testo sull'asse delle y
        
        f"{int(row['games'])}",                 # Parametro s della documentazione ufficiale
                                                # indica il testo da renderizzare
        
        va='center'                             # Parametro va della documentazione ufficiale
                                                # regola l'allineamento verticale del testo
    )

plt.tight_layout()                              # Aggiusta automaticamente i margini della figura per evitare
                                                # che gli elementi del grafico si sovrappongano o vengano tagliati

plt.savefig('figures/dragon_control_rate_per_team.png', dpi=150, bbox_inches='tight')
plt.show()                                      # Renderizza e visualizza tutti i grafici costruiti
                                                # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

L'analisi del *dragon control rate* medio evidenzia differenze significative tra i team nella capacità di controllare gli obiettivi neutrali nel corso delle partite. **Karmine Corp** emerge come il team con il miglior controllo dei draghi, mantenendo in media circa il 59% dei draghi totali per partita, seguito da **G2 Esports** e **Fnatic**, entrambe sopra il 56%. Un secondo gruppo di team — fra cui **Movistar KOI** — mostra valori leggermente superiori alla soglia del 50%, indicativi di una gestione complessivamente equilibrata ma meno dominante rispetto ai team di vertice. **SK Gaming** e **Natus Vincere** presentano invece i valori più bassi, evidenziando una difficoltà strutturale nel competere per questo obiettivo.

Nel complesso, i risultati suggeriscono che il controllo dei draghi nel LEC 2025 sia fortemente concentrato in poche squadre, mentre la maggior parte dei team tende a subire il controllo degli obiettivi piuttosto che imporlo.

**Nota metodologica.** Dal grafico si osserva che Natus Vincere ha disputato un numero significativamente inferiore di partite rispetto agli altri team (8 incontri complessivi). Un approfondimento dei dati mostra che il team è presente esclusivamente nel Summer Split 2025, limitatamente alla regular season, nel periodo compreso tra il 2 e il 26 agosto 2025. Anche **Rogue** e **SK Gaming** presentano un numero di partite inferiore alla media, verosimilmente in conseguenza della mancata partecipazione ai playoff. I risultati relativi a tali team vengono pertanto riportati per completezza informativa, ma devono essere interpretati con cautela, in quanto basati su campioni ridotti e non direttamente comparabili con quelli dei team presenti in un numero maggiore di partite.

## 2.6 Capacità di conversione del primo Baron

**Quali team hanno il miglior *Baron conversion rate* — ovvero la quota più elevata di partite vinte dopo aver ottenuto il primo Baron?**

In [ ]:
baron_conv_df = lec_df[                                         # Creazione del DataFrame 'baron_conv_df' partendo dal Dataframe 'lec_df'                                                     
    ['gameid', 'position', 'teamname', 'firstbaron', 'result']  # Colonne del vecchio DataFrame da tenere in quello nuovo
].copy()                                                        # '.copy()' serve a creare una copia del nuovo dataframe in modo che i due
                                                                # siano indipendenti

baron_conv_df = baron_conv_df[                                  # Filtraggio delle righe del DataFrame 'baron_conv_df'.                       
    baron_conv_df['position'] == 'team'                         # Si mantengono solo le righe che hanno "team" come valore della colonna "position"
].copy()                                                        # '.copy' crea una copia indipendente del DataFrame

baron_conv_df = baron_conv_df[                                  # Filtraggio delle righe del DataFrame 'baron_conv_df'.
    baron_conv_df['firstbaron'] == 1                            # Si mantengono solo le righe che hanno '1' come valore della colonna "firstbaron"
]                                                               

team_baron_conversion = (                                       # Creazione del DataFrame 'team_baron_conversion'
    baron_conv_df                                               # DataFrame di partenza
    .groupby('teamname', as_index=False)                        # '.groupby' raggruppa i dati per i valori della colonna 'teamname'
                                                                # 'as_index = False' riporta la colonna 'teamname' a colonna normale
    .agg(                                                       # '.agg()' permette di effettuare varie operazioni sui dati raggruppati
        games_with_first_baron=('result', 'count'),             # Crea la colonna 'games_with_first_baron' conteggiando i valori della colonna 'result'
        wins_after_first_baron=('result', 'sum')                # Crea la colonna 'wins_after_first_baron' sommando i valori della colonna 'result'
    )
)

team_baron_conversion['baron_conversion_rate'] = (              # Creazione della colonna 'baron_conversion_rate' nel DataFrame 'team_baron_conversion'
    team_baron_conversion['wins_after_first_baron'] /           # i valori della colonna sono dati dal rapporto dei valori delle colonne
    team_baron_conversion['games_with_first_baron']             # 'wins_after_first_baron' e 'games_with_first_baron'
)

team_baron_conversion = team_baron_conversion.sort_values(      # Applicazione della funzione '.sort_values' al DataFrame 'team_baron_conversion'
    'baron_conversion_rate',                                    # che ordina in modo discendente le righe del DataFrame 
    ascending=False                                             # secondo i valori della colonna 'baron_conversion_rate'
)

plt.figure(figsize=(10, 6))                                     # Inizializza l'oggetto 'Figure' cioè la tela su cui verrà renderizzato il grafico
                                                                # e con 'figsize' si imposta la sua grandezza

plt.barh(                                                       # Creazione di un grafico a barre orizzontali
    team_baron_conversion['teamname'],                          # Valori che andranno a comporre l'asse delle y
    team_baron_conversion['baron_conversion_rate']              # Valori che andranno a formare l'asse delle x
)

plt.xlabel('Baron conversion rate (win | first Baron)')         # '.xlabel' aggiunge un'etichetta testuale all'asse delle x

plt.title('Capacità di conversione del primo Baron in vittoria – LEC 2025') # '.title' aggiunge il titolo al grafico

plt.axvline(0.5, linestyle='--')                                # '.axvline' aggiunge una linea verticale sul grafico
                                                                # 0.5 indica il punto da cui tracciare la linea sull'asse delle x
                                                                # 'linestyle' regola lo stile grafico della linea, inq uesto caso sarà
                                                                # una linea tratteggiata

for _, row in team_baron_conversion.iterrows():                 # '.iterrows()' itera il DataFrame riga per riga
                                                                # restituendo per ogni riga:
                                                                # 'index' → indice della riga (omesso con '_')
                                                                # 'data' → dati della riga come Series (associato a 'row')
    
    plt.text(                                                   # '.text' aggiunge del testo al grafico
        row['baron_conversion_rate'] + 0.01,                    # Posizione del testo sull'asse delle x, spostato leggermente verso destra
        row['teamname'],                                        # Posizione del testo sull'asse delle y
        f"{int(row['games_with_first_baron'])}",                # Testo da visualizzare
        va='center'                                             # Allineamento verticale del testo
    )

plt.xlim(0, 1)                                                  # Imposta i limiti massimo e minimo dei valori sull'asse delle x

plt.tight_layout()                                              # Aggiusta automaticamente i margini della figura per evitare
                                                                # che gli elementi del grafico si sovrappongano o vengano tagliati

plt.savefig('figures/baron_conversion_rate_per_team.png', dpi=150, bbox_inches='tight')
plt.show()                                                      # Renderizza e visualizza tutti i grafici costruiti
                                                                # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

Il grafico evidenzia come **G2 Esports**, **Karmine Corp** e **Movistar KOI** siano i team con la maggiore capacità di convertire la conquista del primo Baron Nashor in vittoria. Alcuni team mostrano valori di conversione comparabili o superiori; tuttavia, tali risultati sono basati su un numero ridotto di partite in cui è stato conquistato il primo Baron e devono pertanto essere interpretati con cautela, in quanto meno rappresentativi dal punto di vista statistico.

Complessivamente, il risultato suggerisce che la capacità di capitalizzare il primo Baron rappresenti un elemento distintivo dei team di vertice, pur richiedendo un'adeguata contestualizzazione in presenza di campioni ridotti.

## 2.7 Torri conquistate nei primi 20 minuti

**Quali team prendono più torri nei primi 20 minuti?**

La domanda relativa al numero di torri conquistate nei primi 20 minuti risulta di particolare interesse dal punto di vista dell'analisi dell'*early game*. Tuttavia, il dataset a disposizione fornisce esclusivamente informazioni sul numero totale di torri conquistate a fine partita, senza indicazioni temporali sul momento della loro conquista. In assenza di variabili che permettano di identificare il *timing* degli obiettivi, non è possibile distinguere tra torri ottenute nelle fasi iniziali o avanzate della partita. L'analisi viene dunque omessa al fine di evitare inferenze non supportate dai dati.

## 2.8 Controllo della vision e win rate

**Il controllo della vision (wards piazzate e distrutte) è associato a un win rate più elevato?**

In [ ]:
vision_df = lec_df[                                      # Creazione del DatFrame 'vision_df' partendo dal DataFrame 'lec_df'
    ['gameid', 'position', 'visionscore', 'result']      # Righe che formeranno il DataFrame 'vision_df'
].copy()                                                 # '.copy' crea una copia indipendente del DataFrame

vision_df = vision_df[vision_df['position'] == 'team']   # Filtra le righe del DataFrame 'vision_df' mantenendo quelle che hanno 'team'
                                                         # come valore della colonna 'position'

q1, q2, q3 = vision_df['visionscore'].quantile([0.25, 0.5, 0.75]) 
#'.quantile()' calcola i valori soglia ai percentili specificati:
# 'q1' → valore al 25° percentile (25% dei dati è sotto questo valore)
# 'q2' → valore al 50° percentile (mediana)
# 'q3' → valore al 75° percentile (75% dei dati è sotto questo valore)

vision_df.head()                                         # Mostra le prime 5 righe (valore default) del DataFrame

In [ ]:
vision_df['vision_bin'] = pd.cut(                    # Crea la colonna 'vision_bin' tramte la funzione '.cut()' che divide i dati
                                                     # in intervalli regolari definiti dal parametro 'bin'
    
    vision_df['visionscore'],                        # Indica quale colonna utilizzare per la divisione dei dati
    
    bins=[-float('inf'), q1, q2, q3, float('inf')],  # Il parametro 'bin' definisce gli intervalli da creare
                                                     # '-float(inf)' e 'float(inf)' rappresentano rispettivamente l'infinito negativo e
                                                     # quello positivo, gli intervalli sarano quindi 
                                                     # (-∞,  q1](q1, q2](q2, q3](q3, +∞)
    
    labels=[                                         # Assegna un'etichetta testuale ad ogni intervallo,
        f'≤ {q1:.0f}',                               # Etichetta per i valori inferiori a q1
        f'{q1:.0f} – {q2:.0f}',                      # Etichetta per i valori tra q1 e q2
        f'{q2:.0f} – {q3:.0f}',                      # Etichetta per i valori tra q2 e q3
        f'> {q3:.0f}'                                # Etichetta per i valori superiori a q3
    ]
)

vision_bin_winrate = (                               # Creazione del DataFrame 'vision_bin_winrate'
    vision_df                                        # DataFrame di partenza
    .groupby('vision_bin')                           # Raggruppa i dati per i valori della colonna 'vision_bin'
    .agg(                                            # '.agg()' permette di effetuare varie operazioni sui dati
        win_rate=('result', 'mean'),                 # Crea la colonna 'win_rate' calcolando le medie della colonna 'result'
        games=('result', 'count')                    # Crea la colonna 'games' conteggiando i valori della colonna 'result'
    )
    .reset_index()                                   # Riporta la colonna 'vision_bin' (o qualsiasi colonna su cui è stata utilizzata
                                                                                        # la funzione '.groupby()') a colonna normale
)

vision_bin_winrate                                   # Visualizza il DataFrame 'vision_bin_winrate'


In [ ]:
plt.figure(figsize=(8, 5))                 # Inizializza l'oggetto 'Figure' ossia la 'tela' su cui verrà disegnato il grafico

plt.bar(                                   # Crea un grafico a barre
    vision_bin_winrate['vision_bin'],      # Parametro x nella documentazione ufficiale, indica i dati da inserire nell'asse delle x
    vision_bin_winrate['win_rate']         # Parametro 'heigth' nella documentazione ufficiale
                                           # indica i dati da inserire nell'asse delle y
)

plt.xlabel('Vision score (fasce numeriche)')   # '.xlabel' serve ad impostare un'etichetta testuale per l'asse delle x
plt.ylabel('Win rate')                         # '.ylabel' serve ad impostare un'etichetta testuale per l'asse delle y
plt.title('Win rate per fasce di Vision Score – LEC 2025') # '.title' serve ad impostare il titolo del grafico

plt.ylim(0, 1)                      # 'ylim' imposta i limiti che i valori sull'asse delle y possono avere, in questo caso 0 e 1
plt.axhline(0.5, linestyle='--')    # 'axhline' serve a disegnare una linea orizzontale sul grafico
                                    # 0.5 indica il punto sull'asse delle y da cui partirà la linea
                                    # 'linestyle' imposta lo stile grafico della linea

# Annotazioni: numero di partite per fascia
for i, row in vision_bin_winrate.iterrows():   # '.iterrows()' itera il DataFrame riga per riga
                                               # restituendo per ogni riga:
                                               # 'i' → indice della riga, usato come posizione x del testo
                                               # 'row' → dati della riga come Series
    
    plt.text(                                  # Aggiunge del testo al grafico
        i,                                     # Parametro che indica dove posizionare il testo sull'asse delle x
        row['win_rate'] + 0.02,                # Parametro che indica dove posizionare il testo sull'asse delle y
        int(row['games']),                     # Testo da renderizzare (in questo caso renderizzerà i valori interi della riga
                                               # nella colonna 'games'
        ha='center'                            # Allineamento orizzontale del testo
    )

plt.tight_layout()                             # Aggiusta automaticamente i margini della figura per evitare
                                               # che gli elementi del grafico si sovrappongano o vengano tagliati

plt.savefig('figures/winrate_per_fascia_vision_score.png', dpi=150, bbox_inches='tight')
plt.show()                                     # Renderizza e visualizza tutti i grafici costruiti
                                               # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

Il grafico mostra il win rate associato a diverse fasce numeriche di *vision score*, evidenziando un andamento complessivamente crescente della probabilità di vittoria all'aumentare del controllo della vision. Le partite caratterizzate da un vision score basso (≤ 233) presentano un win rate significativamente inferiore al 50%, suggerendo che un controllo limitato della mappa sia fortemente penalizzante.

Nelle fasce intermedie (233–271 e 271–330) il win rate aumenta progressivamente, superando la soglia del pareggio e raggiungendo il valore massimo nella fascia 271–330. Nella fascia più elevata (> 330) il win rate rimane superiore al 50% ma mostra una lieve flessione rispetto alla fascia immediatamente precedente, indicando che oltre una certa soglia un ulteriore incremento del vision score non si traduce necessariamente in un aumento della probabilità di vittoria. Complessivamente, i risultati indicano una **associazione positiva** tra controllo della vision e successo competitivo, senza tuttavia implicare un rapporto causale diretto.

## 2.9 Durata media delle partite per team

**Quali team giocano mediamente partite più lunghe?**

In [ ]:
avg_teamgame_df = lec_df[                                # Creazione del DataFrame 'avg_teamgame_df' partendo dal DataFrame 'lec_df'
    ['gameid', 'position', 'teamname', 'gamelength']     # Colonne da mantenere nel nuovo DataFrame
].copy()                                                 # '.copy()' crea una copia indipendente del DataFrame appena creato

# Filtriamo solo i team
avg_teamgame_df = avg_teamgame_df[                       # Si filtrano le righe del database che hanno 'team' come valore 
    avg_teamgame_df['position'] == 'team'                # nella colonna 'position'
].copy()                                                 # '.copy()' crea una copia indipendente del DataFrame appena creato

avg_teamgame_df['gamelength_min'] = (                    # Creazione della colonna 'gamelength_min' che conterrà la durata delle partite in minuti
    avg_teamgame_df['gamelength'] / 60                   # Effettua la divisione dei valori della colonna 'gamelength' per 60
)                                                        # Serve a convertire la durata delle partite da secondi a minuti

team_avg_gamelength = (                                  # Crea il DataFrame 'team_avg_gamelength'
    avg_teamgame_df                                      # DataFrame di partenza
    .groupby('teamname', as_index=False)                 # Raggruppa i dati per i valori della colonna 'teamname' 
                                                         # ed evita che la colonna venga utilizzata come indice
    .agg(                                                # '.agg()' permette di eseguire varie operazioni sui valori raggruppati
        avg_game_length_min=('gamelength_min', 'mean'),  # Crea la colonna 'avg_game_length_min' calcolando le medie della colonna 'gamelength_min'
        games=('gameid', 'nunique')                      # Crea la colonna 'games' calcolando i valori unici della colonna 'gameid'
    )
    .sort_values('avg_game_length_min', ascending=False) # Ordina i dati in modo discendente secondo i valori della colonna 'avg_game_length_min'
)

team_avg_gamelength                                      # Mostra il DataFrame 'team_avg_gamelength' 


In [ ]:
plt.figure(figsize=(10, 6))                                   # Inizializza l'oggetto 'Figure' ed imposta la sua grandezza con 'figsize'

plt.barh(                                                     # Creazione di un grafico a barre orizzontali
    team_avg_gamelength['teamname'],                          # Parametro 'y' della documentazione ufficiale, indica i dati che verranno
                                                              # inseriti sull'asse delle y per la creazione del grafico
    
    team_avg_gamelength['avg_game_length_min']                # Parametro 'width' della documentazione ufficiale, indica i dati che verranno
                                                              # inseriti sull'asse delle x per la creazione del grafico
)

plt.xlabel('Durata media partita (minuti)')                   # '.xlabel' inserisce un'etichetta testuale sull'asse delle x
plt.title('Durata media delle partite per Team – LEC 2025')   # '.title' inserisce il titolo del grafico

# Annotazioni: numero di partite
for _, row in team_avg_gamelength.iterrows():                 # '.iterrows' irea il ddataframe riga per riga
                                                              # restituendo per ogni riga
                                                              # '_' -> rappresenterebbe l'indice della riga di solito rappresentato con la variabile 'i'
                                                              #        in questo caso viene omesso dato che non è utile
                                                              # row -> dati della riga come Series
    
    plt.text(                                                 # '.text' aggiunge del testo al grafico
        row['avg_game_length_min'] + 0.2,                     # Indica la posizione sull'asse delle x su cui viene aggiunto il testo
        row['teamname'],                                      # Indica la posizione sull'asse delle y su cui viene aggiunto il testo
        f"{int(row['games'])}",                               # Il testo da aggiungere
        va='center'                                           # Imposta l'allineamento verticale
    )

plt.tight_layout()                                            # Aggiusta automaticamente i margini della figura per evitare
                                                              # che gli elementi del grafico si sovrappongano o vengano tagliati

plt.savefig('figures/durata_media_partite_per_team.png', dpi=150, bbox_inches='tight')
plt.show()                                                    # Renderizza e visualizza tutti i grafici costruiti
                                                              # ma non ancora mostrati fino a questo punto del codice

### Osservazione grafico

La durata media delle partite si colloca, per la quasi totalità dei team, nella fascia compresa tra **30 e 35 minuti**, indicando una sostanziale omogeneità nella lunghezza dei match all'interno del campionato. Le differenze tra i team risultano contenute e non evidenziano scostamenti marcati verso partite significativamente più brevi o più lunghe.

## 2.10 Stile di gioco — aggressività e gestione del rischio

**Quali team hanno uno stile di gioco più aggressivo (kills/min, deaths/min)?**

In [ ]:
aggression_df = lec_df[                                                     # Creazione del DataFrame 'aggression_df' partendo dal DataFrame 'lec_df'
    ['gameid', 'position', 'teamname', 'gamelength', 'team kpm', 'deaths']  # Colonne del DataFrame di partenza che comporranno il nuovo DataFrame
].copy()                                                                    # '.copy' crea una copia indipendente del nuovo DataFrame

aggression_df = aggression_df[                                              # Si filtrano le righe del DataFrame 'aggression_df' 
    aggression_df['position'] == 'team'                                     # mantenendo quelle che hanno 'team' come valore della colonna 'position'
].copy()                                                                    # '.copy' crea una copia indipendente del DataFrame

aggression_df.rename(columns={'team kpm': 'team_kpm'}, inplace=True)        # '.rename' rinomina la colonna 'team kpm' in 'team_kpm'
                                                                            # 'inplace=True' indica di modificare il dataframe corrente 
                                                                            # invece di crearne una copia

aggression_df['gamelength_min'] = aggression_df['gamelength'] / 60          # Crea la colonna 'gamelength_min' dividendo i valori della colonna
                                                                            # 'gamelength' per 60 in modo da convertire la durata delle partite
                                                                            # da secondi a minuti

aggression_df['team_dpm'] = (                                               # Crea la colonna 'team_dpm', che indica il danno per minuto dei team
    aggression_df['deaths'] / aggression_df['gamelength_min']               # il dato si ottiene eseguendo il rapporto dei valori della colonna
                                                                            # 'deaths' per quelli della colonna 'gamelength_min'
)

aggression_df = aggression_df.drop(                                         # '.drop' serve a eliminare le colonne indicate
    columns=['gamelength', 'gamelength_min', 'deaths']                      # dal DataFrame a cui viene applicata
)

team_aggression = (                                                         # Creazione del DataFrame 'team_aggression'
    aggression_df                                                           # DataFrame di partenza
    .groupby('teamname', as_index=False)                                    # '.groupby' raggruppa i dati per i valori ddella colonna 'teamname'
                                                                            # 'as_index=False' fa in modo che la colonna 'teamname' venga tenuta
                                                                            # come colonna normale
    .agg(                                                                   # '.agg' serve per applicare eseguire varie operazioni sui valori raggruppati
        kills_per_min=('team_kpm', 'mean'),                                 # Creazione della colonna 'kills_per_min' eseguendo le medie sui valori
                                                                            # della colonna 'team_kpm'
        deaths_per_min=('team_dpm', 'mean'),                                # Creazione della colonna 'deaths_per_min' eseguendo le medie sui valori
                                                                            # della colonna 'team_dpm'
        games=('gameid', 'nunique')                                         # Creazione della colonna 'games' calcolando i valori unici
                                                                            # della colonna 'gameid'
    )
)
 
team_aggression                                                             # Visualizzazione del DataFrame 'team_aggression'

In [ ]:
teams = team_aggression['teamname'].unique()                           # Crea la variabile 'teams' che contiene una lista di valori
                                                                       # unici della colonna 'teamname'
colors = plt.cm.tab20(np.linspace(0, 1, len(teams)))                   # Crea la variabile 'colors' che contiene una mappa colore creata
                                                                       # tramite '.cm' a cui viene assegnata la reference 'tab20'
                                                                       # '.linespace' genera una sequenza di numeri equidistanti su uno
                                                                       # specifico intervallo definito da:
                                                                       # '0' -> Parametro start, indica l'inizio dell'intervallo
                                                                       # '1' -> Parametro stop, indica la fine dell'intervallo
                                                                       # 'len(teams)' -> Parametro num, indica la grandezza del campione

plt.figure(figsize=(8, 6))                                             # Inizializza l'oggetto figure, cioè la tavola su cui verrà
                                                                       # renderizzato il grafico e ne imposta la grandezza tramite 'figsize'

for team, color in zip(teams, colors):                                 # 'zip()' accoppia gli elementi di 'teams' e 'colors' in base
                                                                       # alla loro posizione e poi li itera tramite un for-loop
    
    team_data = team_aggression[team_aggression['teamname'] == team]   # Filtra il DataFrame mantenendo solo la riga del team corrente
                                                                       # nell'iterazione, il valore di 'team' cambia ad ogni ciclo  
    
    plt.scatter(                                                       # Crea un grafico a dispersione  
        
        team_data['deaths_per_min'],                                   # Parametro x della documentazione ufficiale, indica i dati che verranno
                                                                       # utilizzati sull'asse delle x
        
        team_data['kills_per_min'],                                    # Parametro y della documentazione ufficiale, indica i fati che verranno
                                                                       # utilizzati sull'asse delle y
        
        color=color,                                                   # Parametro color della documentazione ufficiale, gestisce i colori del grafico
        
        label=team                                                     # Parametro label della documentazione ufficiale, inserisce delle etichette
                                                                       # testuali sul grafico
    )

# linee medie (quadranti)
plt.axvline(                                                           # '.axvline' inserisce una linea verticale sul grafico
    
    team_aggression['deaths_per_min'].mean(),                          # Parametro x della documentazione ufficiale, indica in quale punto del grafico
                                                                       # inserire la linea
    
    linestyle='--'                                                     # Indica lo stile grafico della linea
)   

plt.axhline(                                                           # 'axhline' inserisce una linea orizzontale sul grafico
    
    team_aggression['kills_per_min'].mean(),                           # Parametro y della documentazione ufficiale, inidica in quale punto del grafico
                                                                       # inserire la linea
    
    linestyle='--'                                                     # Indica lo stile grafico della linea
)

# etichette
for _, row in team_aggression.iterrows():                              # Iterrows itera le righe del DataFrame come coppia 'indice' 'serie'
                                                                       # in questo caso l'indice viene indicato con '_' perchè è omesso
                                                                       # mentre le serie (cioè le varie informazioni della riga) sono indicate con 'row'
    
    plt.text(                                                          # '.text' inserisce del testo sul grafico
        
        row['deaths_per_min'] + 0.002,                                 # Parametro x della documentazione ufficiale, indica la posizione del testo
                                                                       # sull'asse delle x
        
        row['kills_per_min'] + 0.002,                                  # Parametro y della documentazione ufficiale, indica la posizione del testo
                                                                       # sull'asse delle y
        
        row['teamname'],                                               # Parametro s della documentazione ufficiale, gestisce il testo da inserire
                                                                       # sul grafico
        
        fontsize=9                                                     # 'fontsize' gestisce la grandezza del testo
    )

plt.xlabel('Deaths per minute')                                            # '.xlabel' inserisce un'etichetta testuale sull'asse delle x

plt.ylabel('Kills per minute')                                             # '.ylabel' inserisce un'etichetta testuale sull'asse delle y

plt.title('Stile di gioco dei Team – Aggressività vs Rischio (LEC 2025)')  # '.title' inserisce il titolo del grafico

plt.tight_layout()                                                     # Aggiusta automaticamente i margini della figura per evitare
                                                                       # che gli elementi del grafico si sovrappongano o vengano tagliati

plt.savefig('figures/stile_di_gioco_team_kpm_vs_dpm.png', dpi=150, bbox_inches='tight')
plt.show()                                                             # Renderizza e visualizza tutti i grafici costruiti
                                                                       # ma non ancora mostrati fino a questo punto del codice


### Osservazione grafico

Lo *scatter plot* relaziona i team del LEC 2025 sulla base di due indicatori — uccisioni per minuto (KPM) e morti per minuto (DPM) — suddividendoli in quattro quadranti che consentono di caratterizzare lo stile di gioco in termini di aggressività offensiva e gestione del rischio.

Dall'analisi emerge che i team con il miglior profilo secondo questi parametri — ovvero quelli capaci di mantenere un'elevata pressione offensiva contenendo al tempo stesso il numero di morti — sono **G2 Esports**, **Karmine Corp**, **Fnatic**, **Movistar KOI** e **GiantX**. Tali team mostrano uno stile di gioco proattivo ma controllato, suggestivo di una buona capacità di selezionare e gestire i combattimenti.

**Team Vitality** e **Team BDS** si collocano invece nell'area dei team fortemente aggressivi ma più esposti al rischio: pur mantenendo buoni valori di KPM, presentano anche un elevato numero di morti per minuto, indicativo di una maggiore difficoltà nel contenere le risposte avversarie. I rimanenti team si posizionano in zone caratterizzate da una produzione di uccisioni contenuta e da un'elevata esposizione alle morti, suggerendo uno stile più reattivo o difensivo, in cui tendono a subire l'iniziativa avversaria piuttosto che imporre il proprio ritmo di gioco.

## 2.11 Profilo *low kill — high objective efficiency*

**Esistono team che presentano un profilo *low kill — high objective efficiency*?**

In [ ]:
kpm_df = (                                      # Creazione del DataFrame 'kpm_df'
    
    lec_df[lec_df['position'] == 'team']        # DataFrame di partenza (lec_df) filtrato tramite il valore 'team' della colonna 'position'
    
    .groupby('teamname', as_index=False)        # Raggruppa i dati per i valori della colonna 'teamname', 'as_index = False' fa in modo che la colonna
                                                # venga mantenuta come colonna normale e non utilizzata come indice
    
    .agg(                                       # '.agg' permette di effettuare varie operazioni sui dati raggruppati
    
        kills_per_min=('team kpm', 'mean'),     # Crea la colonna 'kills_per_min' le cui informazioni 
                                                # sono date dalla media dei valori della colonna 'team kpm'
        
        games=('gameid', 'nunique')             # Crea la colonna 'games' le cui informazioni sono date 
                                                # dal conteggio dei valori unici della colonna 'gameid'
    )
)

winrate_df = (                                  # Crea il DataFrame 'winrate_df'
    
    lec_df[lec_df['position'] == 'team']        # DataFrame di partenza (lec_df) filtrato tramite il valore 'team' della colonna 'position' 
    
    .groupby('teamname', as_index=False)        # Raggruppa i dati per i valori della colonna 'teamname', 'as_index = False' fa in modo che la colonna
                                                # venga mantenuta come colonna normale e non utilizzata come indice
    
    .agg(                                       # '.agg' permette di effettuare varie operazioni sui dati raggruppati 
        
        win_rate=('result', 'mean')             # Crea la colonna 'win_rate' le cui informazioni sono date dalla media dei valori della colonna 'result'
    )           
)

dragon_df = lec_df[['gameid', 'position', 'teamname', 'dragons']].copy()   # Creazione del DataFrame 'dragon_df' partendo dal DataFrame 'lec_df' e 
                                                                           # mantenendo colo le colonne indicate.
                                                                           # '.copy' crea una copia indipendente del DataFrame appena creato

dragon_df = dragon_df[dragon_df['position'] == 'team'].copy()              # Filtra le righe del DataFrame 'dragon_df' per il valore 'team'
                                                                           # della colonna 'position'

dragon_df['total_dragons_game'] = dragon_df.groupby('gameid')['dragons'].transform('sum')   # Crea la colonna 'total_dragon_game' nel DataFrame 'dragon_df'
                                                                                            # raggruppando il DataFrame per i valori della colonna 'gameid'
                                                                                            # ed eseguendo la somma dei valori della colonna 'dragons'

dragon_df['dragon_control_rate'] = dragon_df['dragons'] / dragon_df['total_dragons_game']   # Crea la colonna 'dragon_control_rate' eseguendo il rapporto
                                                                                            # tra i valori della colonna 'dragons' e quelli della colonna
                                                                                            # 'total_dragons_game'

dragon_df.loc[dragon_df['total_dragons_game'] == 0, 'dragon_control_rate'] = None      
# '.loc' individua le righe dove 'total_dragons_game' è uguale a 0
# e assegna il valore 'None' alla colonna 'dragon_control_rate' per quelle righe
# evitando valori errati causati da divisioni per zero

dragon_team = (                                                  # Creazione del DataFrame 'dragon_team'
    
    dragon_df                                                    # DataFrame di partenza
    
    .groupby('teamname', as_index=False)                         # Raggruppa i dati per i valori della colonna 'teamname',
                                                                 # 'as_index = False' previene che la colonna 'teamname' venga utilizzata come indice

    .agg(                                                        # '.agg' permette di effettuare operazioni specifiche con i dati
        dragon_control_rate=('dragon_control_rate', 'mean')      # Crea la colonna 'dragon_control_rate' i cui valori sono dati dalla media dei valori
                                                                 # della colonna 'dragon_control_rate' 
                                                                 # (la colonna 'dragon_control_rate' di origine fa riferimento a quella del DataFrame
                                                                 # 'dragon_df')
    )     
)

baron_df = lec_df[['gameid', 'position', 'teamname', 'firstbaron', 'result']].copy()          # Creazione del DataFrame 'baron_df' partendo 
                                                                                              # dal DataFrame 'lec_df' e mantenendo solo le colonne indicate

baron_df = baron_df[(baron_df['position'] == 'team') & (baron_df['firstbaron'] == 1)].copy()  # Filtraggio delle righe di 'baron_df' in base ai valori
                                                                                              # 'team' della colonna 'posirion' e '1' della colonna
                                                                                              # 'firstbaron'

baron_team = (                                                  # Creazione del DataFrame 'baron_team'
    
    baron_df                                                    # DataFrame di partenza
    
    .groupby('teamname', as_index=False)                        # '.groupby' raggruppa i dati per i valori della colonna 'teamname'
                                                                # 'as_index = False' previene che la colonna 'teamname' venga utilizzata come indice
                                                                # e la mantiene come colonna normale
    
    .agg(                                                       # '.agg' permette di effettuare varie operazioni sui dati
        
        games_with_first_baron=('result', 'count'),             # Creazione della colonna 'games_with_first_baron' i cui valori sono dati dal
                                                                # conteggio dei valori della colonna 'result'
        
        wins_after_first_baron=('result', 'sum')                # Creazione della colonna 'wins_after_first_baron' i cui valori sono dati dalla
                                                                # somma dei valori della colonna 'result'
    )
)

baron_team['baron_conversion_rate'] = (                         # Creazione della colonna 'baron_conversion_rate' all'interno del DataFrame                                                                   
    baron_team['wins_after_first_baron'] /                      # 'baron_team', i cui valori sono dati dal rapporto dei valori della colonna
    baron_team['games_with_first_baron']                        # 'wins_after_first_baron' e 'games_with_first_baron'
)

baron_team = baron_team[['teamname', 'baron_conversion_rate', 'games_with_first_baron']] # Aggiornamento del DataFrame 'baron_team' mantenendo solo
                                                                                         # le colonne indicate

team_summary = (                                      # Creazione del DataFrame 'team_summary'
    
    kpm_df                                            # DataFrame di partenza
    
    .merge(                                           # Metodo che unisce il DataFrame indicato a quello di partenza
        winrate_df,                                   # DataFrame da unire
        on='teamname',                                # Colonna con i valori chiave da utilizzare per l'unione
        how='left'                                    # Metodo di unione, utilizza i valori chiave del DataFrame di sinistra
    )     
    .merge(                                           # Metodo che unisce il DataFrame indicato a quello di partenza
        dragon_team,                                  # DataFrame da unire
        on='teamname',                                # Colonna con i valori chiave da utilizzare per l'unione
        how='left'                                    # Metodo di unione, utilizza i valori chiave del DataFrame di sinistra
    )
    .merge(                                           # Metodo che unisce il DataFrame indicato a quello di partenza
        baron_team,                                   # DataFrame da unire
        on='teamname',                                # Colonna con i valori chiave da utilizzare per l'unione
        how='left'                                    # Metodo di unione, utilizza i valori chiave del DataFrame di sinistra
          )
)

team_summary                                          # Mostra il DatFrame


In [ ]:
kpm_mean = team_summary['kills_per_min'].mean()             # Variabile che contiene la media dei valori della colonna "kills_per_min'
dragon_mean = team_summary['dragon_control_rate'].mean()    # Variabile che contiene la media dei valori della colonna 'dragon_control_rate'
baron_mean = team_summary['baron_conversion_rate'].mean()   # Variabile che contiene la media dei valori della colonna 'baron_conversion_rate'
winrate_mean = team_summary['win_rate'].mean()              # Variabile che contiene la media dei valori della colonna 'win_rate'

kpm_mean, dragon_mean, baron_mean, winrate_mean             # Visualizzazione di tutte le variabili

team_summary['objective_efficiency'] = (                            # Creazione della colonna 'objective_efficiency' nel DataFrame 'team_summary'
    team_summary[['dragon_control_rate', 'baron_conversion_rate']]  # Selezione delle colonne 'dragon_control_rate' e 'baron_conversion_rate'
    .mean(axis=1)                                                   # '.mean' esegue la media
)                                                                   # 'axis = 1' imposta il calcolo della media sulla righa invece che sulle colonne

obj_mean = team_summary['objective_efficiency'].mean()              # Creazione delle variabile 'obj_mean' che contiene la media dei valori della colonna
                                                                    # 'objective_efficiency'

obj_mean                                                            # Visualizza la variabile 'obj_mean'

In [ ]:
team_summary['low_kill'] = team_summary['kills_per_min'] < kpm_mean
# Creazione della colonna 'low_kill' nel DataFrame 'team_summary' i cui dati sono valori booleani dati dal confronto dei valori della 
# colonna 'kills_per_min' e la variabile 'kpm_mean'

team_summary['high_obj_eff'] = team_summary['objective_efficiency'] > obj_mean
# Creazione della colonna 'high_obj_eff' nel DataFrame 'team_summary' i cui dati sono valori booleani dati dal confronto dei valori della 
# colonna 'objective efficiency' e la variabile 'obj_mean'

team_summary['above_avg_winrate'] = team_summary['win_rate'] > winrate_mean
# Creazione della colonna 'above_avg_winrate' nel DataFrame 'team_summary' i cui dati sono valori booleani dati dal confronto dei valori della 
# colonna 'win_rate' e la variabile 'winrate_mean'

low_kill_high_eff = team_summary[                             # Creazione del DataFrame 'low_kill_high_eff'
    team_summary['low_kill'] & team_summary['high_obj_eff']   # Filtra le righe dove ENTRAMBE le condizioni sono True:
                                                              # 'low_kill' = True (kills_per_min sotto la media)
                                                              # 'high_obj_eff' = True (objective_efficiency sopra la media)
                                                              # '&' è l'operatore AND vettoriale
].sort_values('objective_efficiency', ascending=False)        # Ordina il risultato in modo discendente per 'objective_efficiency'

low_kill_high_eff                                             # Visualizza il DataFrame 'low_kill_high_eff'


In [ ]:
low_kill_high_eff_winning = team_summary[
    team_summary['low_kill'] & team_summary['high_obj_eff'] & team_summary['above_avg_winrate']
].sort_values('objective_efficiency', ascending=False)
# Creazione del DataFrame 'low_kill_high_eff_winning' partendo dal DataFrame 'team_summary' 
# Filtraggio delle righe dove i valori di 'low_kill', 'hight_obj_eff' e 'above_avg_winrate' sono contemporaneamente TRUE
# '.sort_values' ordina in modo discendente il DataFrame per i valori di 'objective_efficiency'
 
low_kill_high_eff_winning # Visualizza il DataFrame 'low_kill_high_eff_winning'


In [ ]:
plt.figure(figsize=(8, 6))                     # Inizializza l'oggetto 'Figure' cioè la 'tela' su cui verrà disegnato il grafico  

plt.scatter(                                   # Creazione di un grafico a dispersione
    
    team_summary['kills_per_min'],             # Parametro x della documentazione ufficiale, rappresenta i dati che verranno utilizzati
                                               # sull'asse delle x
    
    team_summary['objective_efficiency']       # Parametro y della documentazione ufficiale, rappresenta i dati che verranno utilizzati
                                               # sull'asse delle y
)

# linee medie
plt.axvline(                                   # '.axvline' disegna una linea verticale sul grafico
    kpm_mean,                                  # Punto da cui parte la linea
    linestyle='--'                             # Stile grafico della linea
)         

plt.axhline(                                   # '.axhline' disegna una linea orizzontale sul grafico
    obj_mean,                                  # Punto da cui parte la linea
    linestyle='--'                             # Stile grafico della linea
)

# etichette
for _, r in team_summary.iterrows():           # '.iterrows' crea le variabili 'index' (rappresentata da '_') e Series (rappresentata da 'r')
                                               # e poi li itera tramite ciclo for
                                               # Index: i valori numerici che identificano le righe
                                               # Series: le informazioni della riga come serie
    
    plt.text(                                  # Inserisce del testo sl grafico
        r['kills_per_min'] + 0.002,            # Posizione del testo sull'asse delle x
        r['objective_efficiency'] + 0.002,     # Posizione del testo sull'asse delle y
        r['teamname'],                         # Testo visualizzare
        fontsize=9                             # Grandezza del testo
    )

plt.xlabel('Kills per minute (KPM)')                                         # '.xlabel' aggiunge un'etichetta testuale sull'asse delle x
plt.ylabel('Objective efficiency (mean: dragon control, baron conversion)')  # '.ylabel' aggiunge un'etichetta testuale sull'asse delle y
plt.title('Low-kill vs Objective efficiency – LEC 2025')                     # '.title' aggiunge il titolo al grafico
plt.tight_layout()                                                           # Sistema automaticamente i margini in modo che le parti del
                                                                             # grafico non si sovrappongano tra loro

plt.savefig('figures/low_kill_vs_objective_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()                                                                   # Renderizza tutti i grafici non renderizzati fino a questo punto

### Osservazione grafico

Lo *scatter plot* mette in relazione la produzione media di uccisioni per minuto (KPM) con uno *score di efficienza sugli obiettivi*, calcolato come media tra *dragon control rate* e *first Baron conversion rate*. Le linee tratteggiate rappresentano i valori medi del campionato e suddividono lo spazio in quattro aree interpretative.

Dall'analisi emerge che non esistono veri e propri team *low kill – high objective efficiency* in senso stretto: i team che mostrano un'elevata efficienza nel controllo degli obiettivi tendono infatti ad avere anche valori di KPM pari o superiori alla media del campionato. Ciò suggerisce che, nel contesto del LEC 2025, il controllo macro degli obiettivi principali sia generalmente accompagnato da una pressione offensiva significativa.

Alcuni team si avvicinano tuttavia a questo archetipo. In particolare, **Team Heretics** e **SK Gaming** presentano valori di KPM inferiori alla media ma un'efficienza sugli obiettivi superiore o prossima alla media, indicando uno stile più orientato al controllo macro che al combattimento continuo. **GiantX** rappresenta un caso intermedio, con una produzione di kill moderata e una buona capacità di conversione degli obiettivi.

Al contrario, team come **G2 Esports**, **Fnatic** e **Karmine Corp** si collocano nel quadrante in alto a destra, evidenziando come l'elevata efficienza sugli obiettivi sia accompagnata anche da un'elevata aggressività, piuttosto che da uno stile *low kill*. Il caso di **Natus Vincere** appare come un *outlier*, caratterizzato da bassi valori sia di KPM sia di efficienza sugli obiettivi; tale risultato va tuttavia interpretato con cautela, dato il numero molto ridotto di partite disputate.

# 3. Performance dei giocatori

In questa sezione l'analisi è condotta a livello individuale, con l'obiettivo di identificare i giocatori più rilevanti per ciascun ruolo e i profili statistici associati alla vittoria. Per garantire la rappresentatività dei risultati, vengono considerati esclusivamente i giocatori con un numero minimo di partite disputate.

## 3.1 Migliori giocatori per gold difference @15

**Quali giocatori presentano il miglior *gold difference* medio a 15 minuti nel LEC 2025?**

In [ ]:
# 1) Selezione colonne
player_gold_df = lec_df[['position', 'playername', 'golddiffat15']].copy()  # Creazione del DataFrame 'player_gold_df' 
                                                                            # Partendo dal DataFrame 'lec_df' e mantenendo
                                                                            # da esso solo le colonne indicate
                                                                            # '.copy' crea una copia indipendente del DataFrame

# 2) Filtro: escludo le righe team e tolgo eventuali null
player_gold_df = player_gold_df[                                            # Si applicano alcuni filtri al DataFrame e tutti e tre devono essere 
                                                                            # rispettati a causa dell'operatore logico AND

    (player_gold_df['position'].ne('team')) &                               # Mantiene tutte le righe che non hanno 'team' come valore della colonna
                                                                            # 'position'

    (player_gold_df['playername'].notna()) &                                # Mantiene tutte le righe che non hanno valori nulli 
                                                                            # nella colonna 'playername'

    (player_gold_df['golddiffat15'].notna())                                # Mantiene tutte le righe che non hanno valori nulli 
                                                                            # nella colonna 'golddiffat15'
]

# 3) Aggregazione: media per player
gold15_by_player = (                                                        # Creazione del DataFrame 'gold15_by_player'
    player_gold_df                                                          # DataFrame di partenza
    .groupby('playername', as_index=False)                                  # Raggruppamento dei dati del DataFrame per i valori
                                                                            # della colonna 'playername', 'as_index = False' fa in modo di mantenere
                                                                            # 'playername' come colonna normale e non essere utilizzata come indice
    
    .agg(                                                                   # '.agg' aggrega e permette di effettuare varie operazioni sui dati
        
        games=('golddiffat15', 'size'),                                     # Creazione della colonna 'games' i cui valori sono dati dalla grandezza
                                                                            # della colonna 'golddiffat15'
        
        avg_golddiff15=('golddiffat15', 'mean')                             # Creazione della colonna 'avg_golddiff15' i cui valori sono dati dalle
                                                                            # medie della colonna 'golddiffat15'
    )
    .sort_values('avg_golddiff15', ascending=False)                         # '.sort_values' ordina i dati del DataFrame in modo discendente
                                                                            # in base ai valori della colonna 'avg_golddiff15'
)

MIN_GAMES = 10                                                              # Creazione della variabile 'MIN_GAMES' a cui viene associato il valore 10

gold15_by_player_filtered = gold15_by_player[gold15_by_player['games'] >= MIN_GAMES]  # Creazione del DataFrame 'gold15_by_player_filtered' partendo
                                                                                      # dal DataFrame 'gold15_by_player' e mantenendo solo le righe
                                                                                      # che hanno un valore maggiore o uguale a 10 della colonna 'games'

gold15_by_player_filtered.head(10)                                          # Visualizzazione delle prime 10 righe del DataFrame 
                                                                            # 'gold15_by_player_filtered' tramite la funzione '.head()'


In [ ]:
top_n = 15                                                                         # Creazione della variabile 'top_n' a cui viene associato il valore 15
top_players = gold15_by_player_filtered.head(top_n).sort_values('avg_golddiff15')  # Creazione del DataFrame 'top_player' partendo dal DataFrame
                                                                                   # 'gold15_by_player_filtered', mantenendo solo le prime 15 righe
                                                                                   # e ordinando i dati in base ai valori della colonna 'avg_golddiff15'

plt.figure(figsize=(10, 6))                                                        # Inizializzazione dell'oggetto'Figure', cioè la 'tela' su cui verrà
                                                                                   # renderizzato il grafico

plt.barh(                            # Creazione di un grafico a barre orizzontali
    top_players['playername'],       # Variabile y della documentazione ufficiale, indica i dati che verranno utilizzati sull'asse delle y
    
    top_players['avg_golddiff15']    # Variabile 'width' della documentazione ufficiale, indica i dati che verranno utilizzati sull'asse delle x
)

plt.xlabel('Gold Difference @15 (media)')   # '.xlabel' aggiunge un'etichetta testuale sull'asse delle x
plt.ylabel('Giocatore')                     # '.ylabel' aggiunge un'etichetta testuale sull'asse delle y
plt.title(f'Top {top_n} giocatori per Gold Diff @15 (min {MIN_GAMES} game)')   # '.title' aggiunge un titolo al grafico
plt.tight_layout()                          # Sistema automaticamente le distanze in moddo che gli elementi del grafico non si sovrappongano
plt.savefig('figures/top15_giocatori_gold_diff_15.png', dpi=150, bbox_inches='tight')
plt.show()                                  # Renderizza tutti i grafici non ancora renderizzati

### Osservazione grafico

Il grafico mostra i 15 giocatori con il *gold difference* medio a 15 minuti più elevato del LEC 2025, considerando esclusivamente i giocatori con almeno 10 partite disputate.

Dall'analisi emerge chiaramente che **SkewMond** è il giocatore con il valore medio più elevato, distinguendosi nettamente dagli altri player presenti in classifica. Tale risultato suggerisce una forte capacità di generare vantaggio economico nella fase di early game, sia a livello individuale sia in termini di gestione della propria lane. I rimanenti giocatori mostrano valori positivi ma più ravvicinati tra loro, suggestivi di performance early game solide ma meno dominanti.

## 3.2 Jungler con la più alta Kill Participation

**Quali jungler partecipano alla percentuale più alta di kill del team (KP%)?**

In [ ]:
# 1) Selezione colonne
jngKP_df = lec_df[['gameid', 'position', 'side', 'teamname', 'playername', 'kills', 'assists']].copy()
# Creazione del DataFrame 'jngKP_df' partendo dal DataFrame 'lec_df' e mantenendo solo le colonne indicate,
# '.copy' crea una copia indipendente del DataFrame

# 2) Filtro: solo righe team e jungler + drop dei null sulle chiavi
jngKP_df = jngKP_df[                               # Filtra le righe del DataFrame 'jngKP_df'
    jngKP_df['position'].isin(['team', 'jng'])     # mantenendo quelle che hanno come valori 'team' e 'jng' nella colonna 'position'
].dropna(subset=['gameid', 'side'])                # '.dropna' indica di eliminare le righe che hanno valori nulli tenendo in considerazione
                                                   # anche le colonne 'gameid' e 'side'

# 3) Team kills per partita e per side
team_kills = (                                     # Creazione del DataFrame 'team_kills'
    jngKP_df[jngKP_df['position'] == 'team']       # Si filtrano le righe che hanno 'team' come valore della colonna 'position'
    [['gameid', 'side', 'teamname', 'kills']]      # Righe da mantenere nel DataFrame 'team_kills'
    .rename(columns={'kills': 'team_kills'})       # '.rename' rinomina la colonna 'kills' in 'team_kills'
)

# 4) Stats jungler per partita e per side
jungler_stats = (                                  # Creazione del DataFrame 'jungler_stats'
    jngKP_df[jngKP_df['position'] == 'jng']        # Si filtrano le righe che hanno 'jng' come valore della colonna 'position'
    [['gameid', 'side', 'teamname', 'playername', 'kills', 'assists']]   # Righe da mantenere nel DataFrame 'jungler_stats'
    .dropna(subset=['playername', 'kills', 'assists']) # '.dropna' indica di eliminare righe che hanno valori nulli tenendo in considerazione
                                                       # anche le colonne 'playername' 'kills' e 'assist'        
)

jungler_stats['jng_kp_events'] = jungler_stats['kills'] + jungler_stats['assists']
# Crazione della colonna 'jng_kp_events' nel DataFrame 'jungler_stats' i cui valori sono dati dalla somma dei valori delle colonne 'kills' e 'assists'

# 5) Merge corretto: gameid + side (così ogni jungler prende SOLO il suo team)

jngKP_merged = jungler_stats.merge(                # Creazione del DataFrame 'jngKP_merged' unendo i DataFrames 'jungler_stats' e 'team_kills'
    team_kills[['gameid', 'side', 'team_kills']],  # Colonne di 'team_kills' da utilizzare per l'unione
    on=['gameid', 'side'],                         # Colonne chiave su cui effettuare l'unione
    how='inner'                                    # Tipo di unione da effettuare
)

# 6) KP% per partita (gestione team_kills = 0)
jngKP_merged['kp_pct'] = (jngKP_merged['jng_kp_events'] / jngKP_merged['team_kills']) * 100  # Creazione della colonna 'kp_pct' 
                                                                                             # nel DataFrame 'jngKP_merged' i cui valori sono dati dal
                                                                                             # rapporto dei valori delle colonne 'jng_kp_events' e
                                                                                             # 'team_kills' e il risultato moltiplicato per 100

jngKP_merged.loc[jngKP_merged['team_kills'] == 0, 'kp_pct'] = pd.NA    # '.loc' individua le righe dove 'team_kills' è uguale a 0
                                                                       # e assegna 'pd.NA' alla colonna 'kp_pct' per quelle righe,
                                                                       # evitando valori errati causati da divisioni per zero

# (Check consigliato) quante righe per gameid? Idealmente ~2 (un jng per team)
# print(jngKP_merged.groupby('gameid').size().value_counts().head())

# 7) Media KP% per jungler
jungler_kp_avg = (                               # Creazione del DataFrame 'jungler_kp_avg'
    
    jngKP_merged.dropna(subset=['kp_pct'])       # partendo dal DataFrame 'jngKP_merged' ed eliminando i valori nulli della colonna 'kp_pct'
    
    .groupby('playername', as_index=False)       # '.groupby' raggruppa i dati in base ai valori della colonna 'playername' e 'as_index = False'
                                                 # fa in modo che la colonna 'playername' non venga utilizzata come indice e mantenga il suo stato di
                                                 # colonna normale
    
    .agg(                                        # '.agg' aggrega i dati e permette di effettuare varie operazioni su di essi        
        games=('kp_pct', 'size'),                # Creazione della colonna 'games' applicando la funzione 'size' (conta tutte le righe della colonna)
                                                 # alla colonna 'kp_pct'
        
        avg_kp_pct=('kp_pct', 'mean')            # Creazione della colonna 'avg_kp_pct' i cui valori sono dati dalle medie dei valori della colonna
                                                 # 'kp_pct'
    )
    .sort_values('avg_kp_pct', ascending=False)  # Ordina i dati del DataFrame in modo discendente in base ai valori della colonna 'avg_kp_pct'
)

# 8) (Opzionale) filtro minimo partite
MIN_GAMES = 10                                   # Crazione della variabile 'MIN_GAMES' ed assegnazione del valore 10 
jungler_kp_avg_filtered = jungler_kp_avg[jungler_kp_avg['games'] >= MIN_GAMES]  # Creazione del DataFrame 'jungler_kp_avg_filtered' le cui righe sono
                                                                                # quelle del DataFrame 'jungler_kp_avg' filtrate per i valori
                                                                                # maggiori o uguali a 10 della colonna 'games'

jungler_kp_avg_filtered.head(10)                                                # Mostra le prime 10 righe del DataFrame 'jungler_kp_avg_filtered'

In [ ]:
# Top N da mostrare
top_n = 15   # Creazione della variabile 'top_n' e assegnazione del valore 15 ad essa

# (assumo tu stia usando il df filtrato)
plot_df = jungler_kp_avg_filtered.head(top_n).sort_values('avg_kp_pct') # Crea il DataFrame 'plot_df' dalle prime 15 righe 
                                                                        # del DataFrame 'jungler_kp_avg_filtered' ordinate in modo ascendente per
                                                                        # i valori della colonna 'avg_kp_pct'

plt.figure(figsize=(10, 6))            # Inizializzazione dell'oggetto 'Figure' ovvero la 'tela' su cui verrà renderizzato il grafico

plt.barh(                              # Creazione di un grafico a barre orizzontali
    plot_df['playername'],             # Parameto y della documentazione ufficiale, rappresenta i dati che verranno utilizzati sull'asse delle y
    plot_df['avg_kp_pct']              # Parametro 'width' della documentazione ufficiale, rappresenta i dati che verranno utilizzati sull'asse delle x
)
plt.xlabel('Kill Participation % (media)')   # '.xlabel' aggiunge un'etichetta testuale all'asse delle x
plt.ylabel('Jungler')                        # '.ylabel' aggiunge un'etichetta testuale all'asse delle y
plt.title(f'Top {top_n} jungler per KP% (min {MIN_GAMES} game)')   # '.title' aggiunge il titolo al grafico
plt.tight_layout()   # Sistema automaticamente i bordi degli elementi del grafico in modo che non si sovrappongano tra loro
plt.savefig('figures/top15_jungler_kill_participation.png', dpi=150, bbox_inches='tight')
plt.show()           # Renderizza tutti i grafici che non sono stati renderizzati fino a questo momento

### Osservazione grafico

Il grafico mostra i 15 jungler con la *Kill Participation* (KP%) media più elevata del LEC 2025, fra i giocatori con almeno 10 partite disputate.

**Malrang** e **ISMA** risultano i jungler con il KP% medio più alto, attestandosi attorno all'80% e indicando un coinvolgimento estremamente elevato nelle uccisioni del proprio team. Tale dato è suggestivo di uno stile di gioco fortemente proattivo, orientato a *gank*, *skirmish* e a una presenza costante nelle azioni decisive. Immediatamente alle loro spalle si collocano **113**, **Razork** e **Lyncas**, con valori comunque molto elevati e relativamente ravvicinati, a conferma del ruolo centrale del jungler nel coordinamento delle azioni di squadra.

Nel complesso, la distribuzione dei valori evidenzia come i jungler con KP% più alto siano generalmente quelli maggiormente coinvolti nelle fasi chiave della partita, rafforzando l'importanza del ruolo nel determinare il ritmo di gioco.

## 3.3 Damage share medio per ruolo

**Quale ruolo presenta il *damage share* medio più elevato?**

In [ ]:
# 1) Selezione colonne
avg_damageshare_df = lec_df[['gameid', 'position', 'damageshare']].copy()   # Creazione del DataFrame 'avg_damageshare_df' partendo dal DataFrame
                                                                            # 'lec_df'e mantenendo solo le colonne selezionate.
                                                                            # '.copy' crea una copia indipenddente del DataFrame

# 2) Filtro: rimuovo team e NaN
avg_damageshare_df = avg_damageshare_df[            # Si filtrano le righe del DataFrame 'avg_damageshare_df'
    (avg_damageshare_df['position'] != 'team') &    # mantenendo solo quelle che hanno un valore diverso da 'team' per la colonna 'position'
    (avg_damageshare_df['damageshare'].notna())     # e quelle che non hanno valori nulli nella colonna 'damageshare'
]

# 3) Check: ogni ruolo deve comparire 2 volte per gameid
role_check = (                          # Creazione del DataFrame 'role_check'
    avg_damageshare_df                  # DataFrame di partenza
    .groupby(['gameid', 'position'])    # Raggruppamento dei dati per i valori delle colonne 'gameid' e 'position'
    .size()                             # '.size' conta il numero di righe pe rogni colonna
)

# Verifica se tutte le combinazioni hanno valore 2
if not (role_check == 2).all():
    pass  # check superato
else:
    pass  # check superato

# 4) Media damage share per ruolo
damage_share_by_role = (                           # Creazione del DataFrame 'damage_share_by_role'
    avg_damageshare_df                             # DataFrame di partenza
    .groupby('position', as_index=False)           # Raggruppamento dei dati per i valori della colonna 'position'
                                                   # 'as_index = False' fa in modo che 'position' mantenga il suo stato di colonna normale e non venga
                                                   # utilizzata come indice
    .agg(                                          # '.agg' aggrega i dati e permette di eseguire varie operazioni su di essi
        avg_damageshare=('damageshare', 'mean'),   # Creazione della colonna 'avg_damageshare' i cui valori sono le medie della colonna 'damageshare'
        games=('damageshare', 'size')              # Creazione della colonna 'games' contando il numrto di righe della colonna 'damageshare'
    )
    .sort_values('avg_damageshare', ascending=False) # 'ascending = False' ordina i dati del DataFrame per i valori della colonna 'avg_damageshare'
                                                     # in modo discendente
)

damage_share_by_role   # Visualizza il DataFrame 'damage_share_by_role'

In [ ]:
plt.figure(figsize=(8,5))                         # Inizializzazione dell'oggetto 'Figure' ossia la 'tela' su cui verrà renderizzato il grafico

plt.bar(                                          # Creazione di un grafico a barre
    damage_share_by_role['position'],             # Variabile x della documentazione ufficiale, indica i dati che verranno utilizzati sull'asse delle x   
        damage_share_by_role['avg_damageshare']   # Variabile 'height' della documentazione ufficiale, indica i dati che verranno utilizzati 
                                                  # sull'asse delle y
)

plt.ylabel('Damage Share Medio')   # '.ylabel' aggiunge un'etichetta testuale all'asse delle y
plt.xlabel('Ruolo')                # '.xlabel' aggiunge un'etichetta testuale all'asse delle x
plt.title('Damage Share medio per ruolo - LEC 2025')  # Aggiunge il titolo al grafico
plt.tight_layout()   # Sistema automaticamente i bordi degli elementi del grafico in modo che non si sovrappongano tra loro
plt.savefig('figures/damage_share_medio_per_ruolo.png', dpi=150, bbox_inches='tight')
plt.show()           # Renderizza tutti i grafici che non sono stati renderizzati fino a questo momento

### Osservazione grafico

Dall'analisi emerge una marcata gerarchia del *damage share* medio per ruolo:

- **Bot (ADC)** è il ruolo con il damage share medio più alto (~27–28%)
- **Mid** segue molto da vicino (~26%)
- **Top** si attesta intorno al ~22%
- **Jungle** contribuisce per circa il ~16%
- **Support** è il ruolo con il damage share medio più basso (~7–8%)

Combinando Bot e Mid si osserva che i due ruoli centrali nel *damage output* del team generano oltre il **50% del danno totale complessivo**, confermando la centralità dei carry nella distribuzione delle risorse e nelle fasi di *teamfight*. Il risultato è coerente con la struttura macro del gioco competitivo, nel quale ADC e Mid rappresentano le principali fonti di danno, mentre Support e Jungle assolvono prevalentemente a funzioni di utility e di controllo della mappa.

## 3.4 Migliori support per Vision Score per Minute

**Quali support presentano il miglior *Vision Score per Minute* (VSPM)?**

In [ ]:
# 1) Selezione colonne
vspm_df = lec_df[['gameid', 'position', 'playername', 'vspm']].copy()  # Creazione del DataFrame 'vspm_df' partendo dal DataFrame 'lec_df'
                                                                       # e mantenendo solo le colonne indicate, '.copy' vrea una copia
                                                                       # indipendente del DataFrame 

# 2) Filtro solo support + rimozione NaN essenziali

vspm_df = vspm_df[                      # Filtra le righe del DataFrame, soddisfacendo tutte le condizioni tramite l'operatore AND (&)
    (vspm_df['position'] == 'sup') &    # Prima condizione: I valori della colonna 'position' devono essere 'sup'
    (vspm_df['playername'].notna()) &   # Seconda condizione: I valori della colonna 'playername' non devono essere vuoti o mancanti
    (vspm_df['vspm'].notna())           # Terza condizione: I valori della colonna 'vspm' non devono essere vuoti o mancanti
]

# 3) Check: ogni gameid deve comparire 2 volte (2 support per partita)
game_check = vspm_df.groupby('gameid').size()

if not (game_check == 2).all():
    pass  # check superato
    print(game_check.value_counts().head(10))
else:
    pass  # check superato

# 4) Media VSPM per support
vspm_by_support = (                           # Creazione del DataFrame 'vspm_by_support'
    vspm_df                                   # DataFrame di partenza
    .groupby('playername', as_index=False)    # Raggruppa i dati in base ai valori della colonna 'playername', 'as_index=False' 
                                              # fa in modo che la colonna 'playername' non diventi un indice e rimanga come colonna normale
    .agg(                                     # '.agg' permette di effettuare varie operazioni sui dati
        games=('vspm', 'size'),               # Creazione della colonna 'games' i cui valori sono dati dal conteggio delle righe 
                                              # della colonna 'vspm'
        avg_vspm=('vspm', 'mean')             # Creazione della colonna 'avg_vspm' i cui valori sono dati dalle medie dei valori della 
                                              # colonna 'vspm'
    )
    .sort_values('avg_vspm', ascending=False) # Ordina il DataFrame in base ai valori della colonna 'avg_vspm' in modo discendente
)

# 5) (Consigliato) filtro minimo partite
MIN_GAMES = 10       # Creazione della variabile 'MIN_GAMES' e assegnazione del valore 10
vspm_by_support_filtered = vspm_by_support[vspm_by_support['games'] >= MIN_GAMES] # Creazione di un DataFrame filtrato in cui ci i valori
                                                                                  # della colonna 'games' sono maggiori o uguali a 10

vspm_by_support_filtered.head(10)   # Mostra le prime 10 righe della DataFrame


In [ ]:
top_n = 15   # Creazione della variabile 'top_n' e assegnazione del valore 15

plot_df = vspm_by_support_filtered.head(top_n).sort_values('avg_vspm') # Creazione del DataFrame 'plot_df' che contiene le prime 15 righe
                                                                       # del DataFrame 'vspm_by_filtered' ordinate in modo ascendente per
                                                                       # i valori della colonna 'avg_vspm'

plt.figure(figsize=(10, 6))                                            # Inizializzazione dell'oggetto 'Figure' ovvero la 'tela' su cui 
                                                                       # verrà renderizzato il grafico

plt.barh(                    # Creazione di un grafico a barre orizzontali
    plot_df['playername'],   # Parametro y della documentazione ufficiale, indica i dati che verranno utilizzati sull'asse delle y
    plot_df['avg_vspm']      # Parametro 'width' della documentazione ufficiale, indica i dati che verranno utilizzati sull'asse delle x
)                   
plt.xlabel('Vision Score per Minute (media)')   # Aggiunge un'etichetta testuale all'asse delle x
plt.ylabel('Support')   # Aggiunge un'etichetta testuale all'asse delle y 
plt.title(f'Top {top_n} support per VSPM (min {MIN_GAMES} game)') # Aggiunge il titolo al grafico
plt.tight_layout() # Sistema automaticamente i margini dei componenti del grafico in modo che non si sovrappongano
plt.savefig('figures/top15_support_vspm.png', dpi=150, bbox_inches='tight')
plt.show()   # Renderizza tutti i grafici non renderizzati fino a questo momento

### Osservazione grafico

Il grafico mostra i 15 support con il *Vision Score per Minute* (VSPM) medio più elevato del LEC 2025, considerando i giocatori con almeno 10 partite disputate. L'analisi evidenzia tre fasce di performance.

**Fascia alta (> 4 VSPM).** Solo **Fleshy** supera la soglia di 4, distinguendosi nettamente dagli altri giocatori. Il dato suggerisce una presenza particolarmente intensa nella gestione della visione e nel controllo della mappa, configurandolo come il support statisticamente più performante in termini di VSPM medio.

**Fascia intermedia (3,5–4 VSPM).** Comprende la maggior parte dei giocatori in classifica — Milkyx, Alvaro, Parus, Stend, Targamas, Execute, Jun e Nisqy — con valori relativamente ravvicinati, indicativi di uno standard competitivo elevato e omogeneo tra i top support del campionato.

**Fascia inferiore (3,0–3,5 VSPM).** Include i support con il VSPM medio più basso fra quelli analizzati (Labrov, Loopy, Malrang, Hylissang). Sebbene collocati nella parte bassa della classifica, i valori restano coerenti con il ruolo e non indicano necessariamente una performance complessivamente insufficiente.

Particolarmente interessante è la presenza di **Labrov** nella fascia più bassa, nonostante appartenga a uno dei team più performanti del campionato. Tale evidenza suggerisce che il controllo della visione possa essere distribuito in modo più collettivo all'interno della squadra, oppure che il team adotti uno stile di gioco meno dipendente dal VSPM individuale del support.

## 3.5 Correlazione tra KDA individuale e win rate

**Esiste una correlazione tra KDA individuale e win rate?**

In [ ]:
# 1) Selezione colonne
kda_df = lec_df[['playername', 'position', 'kills', 'deaths', 'assists', 'result']].copy()
# Creazione del DataFrame 'kda_df' partendo dal DataFrame 'lec_df' e mantenendo solo le colonne indicate
# '.copy' crea una copia indipendente del DataFrame

# 2) Filtro solo player (no team)
kda_df = kda_df[kda_df['position'] != 'team']
# Filtra le righe del DataFrame mantenendo solo quelle che non hanno 'team' come valore di 'position'

# 3) Calcolo KDA per partita
kda_df['kda'] = (kda_df['kills'] + kda_df['assists']) / kda_df['deaths'].replace(0, np.nan)
# Sostituisce i valori 0 della colonna 'deaths' con NaN prima della divisione,
# evitando divisioni per zero che produrrebbero valori infiniti

# 4) Aggregazione per player
player_kda_wr = (     # Creazione del DataFrame 'player_kda_wr'
    kda_df            # DataFrame di partenza
    .groupby('playername', as_index=False) # '.groupby' raggruppa i dati per i valori della colonna 'playername'
                                           # as_index=False' fa in modo che la colonna 'playername' non diventi un indice 
                                           # e rimanga come colonna normale
    .agg(                            # '.agg' aggrega i dati e permetti di effettuare varie operazioni su di essi
        games=('kda', 'size'),       # Creazione della colonna 'games' contando le righe della colonna 'kda'
        avg_kda=('kda', 'mean'),     # Creazione della colonna 'avg_kda' i cui valori sono dati dalle medie dei valori della colonna 'kda'
        win_rate=('result', 'mean')  # Creazione della colonna 'win_rate' i cui valori sono dati dalle medie dei valori della colonna 'result'
    )
)

# 5) Filtro minimo partite
MIN_GAMES = 10   # Creazione della variabile 'MIN_GAMES' e assegnazione del valore 10 
player_kda_wr = player_kda_wr[player_kda_wr['games'] >= MIN_GAMES] # Creazione di un DataFrame filtrato in cui vengono tenute solo le righe
                                                                   # con valore della colonna 'games' maggiore o uguale a 10

player_kda_wr.head()   # Mostra le prime 5 righe del DataFrame


In [ ]:
correlation = player_kda_wr['avg_kda'].corr(player_kda_wr['win_rate'])
# Creazione della variabile 'correlation' a cui verrà associato il valore della correlazione dei dati delle colonne
# 'avg_kda' e 'win_rate' calcolato tramite il metodo pandas '.corr'

print("Correlazione KDA vs Win Rate:", round(correlation, 3))
# Restituisce un stringa di testo e il valore della variabile 'correlation' arrotondato alla terza cifra decimale

In [ ]:
plt.figure(figsize=(8,6)) # Inizializzazione dell'oggetto 'Figure'
plt.scatter(  # Creazione di un grafico a dispersione
    player_kda_wr['avg_kda'], # Parametro x della documentazione ufficiale, indica i dati che verranno utilizzati sull'asse delle x
    player_kda_wr['win_rate'] # Parametro y della documentazione ufficiale, indica i dati che verranno utilizzati sull'asse delle y
)

plt.xlabel("KDA medio")   # Aggiunge un'etichetta testuale all'asse delle x
plt.ylabel("Win Rate")    # Aggiunge un'etichetta testuale all'asse delle y
plt.title("Relazione tra KDA medio e Win Rate - LEC 2025")   # Aggiunge un titolo al grafico

# Inserimento valore correlazione nel grafico
plt.text(                               # Aggiunge del testo al grafico
    0.05, 0.95,                         # Coordinate x e y in sistema 'transAxes',
                                        # posiziona il testo in alto a sinistra
    f"Pearson r = {correlation:.3f}",   # f-string che mostra il coefficiente di correlazione
                                        # di Pearson arrotondato a 3 cifre decimali
    transform=plt.gca().transAxes,      # 'plt.gca()' restituisce l'oggetto 'ax' corrente,
                                        # '.transAxes' imposta il sistema di coordinate relativo al grafico
    fontsize=11,                        # Grandezza del testo
    verticalalignment='top'             # Allineamento verticale del testo dal bordo superiore
)


plt.tight_layout()                  # Sistema automaticamente i margini dei componenti del grafico in modo che non si sovrappongano
plt.savefig('figures/scatter_kda_vs_winrate.png', dpi=150, bbox_inches='tight')
plt.show()                          # Renderizza tutti i grafici non renderizzati fino a questo momento

### Osservazione grafico

L'analisi evidenzia una **correlazione positiva e forte** (r = 0,757) tra il KDA medio individuale e il win rate dei giocatori nel LEC 2025. Lo *scatter plot* mostra una chiara tendenza crescente: all'aumentare del KDA medio cresce anche la percentuale di vittorie, suggerendo che i giocatori con performance individuali più efficienti in termini di rapporto fra kill/assist e morti tendano ad appartenere a squadre più vincenti. Il valore della correlazione rientra nella fascia delle relazioni *forti*, indicativa di un'associazione lineare significativa tra le due variabili.

È tuttavia opportuno sottolineare che **la correlazione non implica causalità**: un KDA elevato può essere tanto una conseguenza del fatto che il team sta vincendo (meno morti, più assist), quanto un contributo attivo alla vittoria. In altri termini, KDA e win rate risultano fortemente associati ma non è possibile affermare che l'uno causi direttamente l'altro.

## 3.6 Impatto dei giocatori nei primi 15 minuti

**Quali giocatori hanno il maggiore impatto nei primi 15 minuti?**

### Osservazione grafico

La domanda è stata esclusa dall'analisi in quanto il concetto di *impatto nei primi 15 minuti* richiede una definizione operativa esplicita — quale, ad esempio, il *gold difference* a 15 minuti, l'*early Kill Participation* o una combinazione ponderata di tali grandezze — che non risultava specificata nella richiesta. Si rinvia pertanto a successive analisi mirate, con definizione preliminare della metrica di riferimento.

## 3.7 Damage share individuale e vittorie

**I giocatori con un *damage share* più alto vincono più partite?**

In [ ]:
# 1) Selezione colonne
sharedmgplayer_df = lec_df[['playername', 'position', 'result', 'damageshare']].copy()

# 2) Filtro: escludo team + rimuovo NaN essenziali
sharedmgplayer_df = sharedmgplayer_df[
    (sharedmgplayer_df['position'] != 'team') &
    (sharedmgplayer_df['playername'].notna()) &
    (sharedmgplayer_df['damageshare'].notna()) &
    (sharedmgplayer_df['result'].notna())
]

# 3) Aggrego per player: avg damage share + win rate + numero partite
player_dmg_wr = (
    sharedmgplayer_df
    .groupby('playername', as_index=False)
    .agg(
        games=('damageshare', 'size'),
        avg_damageshare=('damageshare', 'mean'),
        win_rate=('result', 'mean')
    )
)

# 4) Filtro minimo partite
MIN_GAMES = 10
player_dmg_wr = player_dmg_wr[player_dmg_wr['games'] >= MIN_GAMES].copy()

# 5) Correlazione (Pearson)
corr = player_dmg_wr['avg_damageshare'].corr(player_dmg_wr['win_rate'])
print(f"Correlazione Damage Share vs Win Rate (Pearson r): {corr:.3f}")

# 6) Scatterplot con annotazione correlazione
plt.figure(figsize=(8, 6))
plt.scatter(player_dmg_wr['avg_damageshare'], player_dmg_wr['win_rate'])

plt.xlabel("Damage Share medio")
plt.ylabel("Win Rate")
plt.title("Relazione tra Damage Share medio e Win Rate - LEC 2025")

plt.text(
    0.05, 0.95,
    f"Pearson r = {corr:.3f}",
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment='top'
)

plt.tight_layout()
plt.savefig('figures/scatter_damageshare_vs_winrate.png', dpi=150, bbox_inches='tight')
plt.show()

### Osservazione grafico

L'analisi evidenzia una **correlazione praticamente nulla** (r = 0,012) tra *damage share* medio individuale e win rate nel LEC 2025. Lo *scatter plot* non mostra alcuna tendenza chiaramente crescente o decrescente: giocatori con damage share elevato possono presentare win rate sia alti sia bassi, e la stessa eterogeneità si osserva per i giocatori con damage share contenuto. Tale risultato suggerisce che, in termini aggregati, **un damage share medio più elevato non è associato a una maggiore probabilità di vittoria**.

## 3.8 Stabilità delle performance individuali

**Quali giocatori mantengono performance stabili per tutta la stagione?**

In [ ]:
# 1) Selezione colonne
stability_df = lec_df[['playername', 'position', 'kills', 'deaths', 'assists']].copy()

# 2) Escludo team
stability_df = stability_df[stability_df['position'] != 'team']

# 3) Calcolo KDA per partita
stability_df['kda'] = (
    (stability_df['kills'] + stability_df['assists']) /
    stability_df['deaths'].replace(0, np.nan)
)

# 4) Aggregazione per player
player_stability = (
    stability_df
    .groupby('playername', as_index=False)
    .agg(
        games=('kda', 'size'),
        avg_kda=('kda', 'mean'),
        std_kda=('kda', 'std')
    )
)

# 5) Filtro minimo partite
MIN_GAMES = 10
player_stability = player_stability[player_stability['games'] >= MIN_GAMES]

# 6) Calcolo coefficiente di variazione
player_stability['cv_kda'] = (
    player_stability['std_kda'] /
    player_stability['avg_kda']
)

# 7) Calcolo mediana KDA
median_kda = player_stability['avg_kda'].median()

# 8) Divisione gruppi
stable_positive = (
    player_stability[player_stability['avg_kda'] >= median_kda]
    .sort_values('cv_kda')
)

stable_negative = (
    player_stability[player_stability['avg_kda'] < median_kda]
    .sort_values('cv_kda')
)

print("Top 5 stabilmente forti:")
print(stable_positive.head())

print("\nTop 5 stabilmente deboli:")
print(stable_negative.head())

In [ ]:
from matplotlib.lines import Line2D

df = player_stability.copy()

median_kda = df['avg_kda'].median()
median_std = df['std_kda'].median()

# Palette Okabe-Ito (color-blind friendly)
quad_colors = {
    "Forti & stabili":  "#0072B2",  # blu
    "Forti & volatili": "#E69F00",  # arancione
    "Deboli & stabili": "#009E73",  # verde acqua
    "Deboli & volatili":"#CC79A7",  # viola
}

def get_quadrant(row):
    if row['avg_kda'] >= median_kda and row['std_kda'] <= median_std:
        return "Forti & stabili"
    elif row['avg_kda'] >= median_kda and row['std_kda'] > median_std:
        return "Forti & volatili"
    elif row['avg_kda'] < median_kda and row['std_kda'] <= median_std:
        return "Deboli & stabili"
    else:
        return "Deboli & volatili"

df['quadrant'] = df.apply(get_quadrant, axis=1)
df['color'] = df['quadrant'].map(quad_colors)

# --- PLOT ---
plt.figure(figsize=(12, 6))
ax = plt.gca()

# Scatter con bordo nero (migliora leggibilità)
ax.scatter(
    df['avg_kda'],
    df['std_kda'],
    c=df['color'],
    edgecolors='black',
    linewidths=0.3
)

# Linee mediane
ax.axvline(median_kda, linestyle='--')
ax.axhline(median_std, linestyle='--')

ax.set_xlabel("KDA medio")
ax.set_ylabel("Deviazione standard KDA")
ax.set_title("Stabilità performance (KDA): media vs variabilità - LEC 2025")

# Etichette quadranti (discrete)
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
dx = (xmax - xmin) * 0.03
dy = (ymax - ymin) * 0.05

ax.text(xmax - dx, ymin + dy, "Forti & stabili", ha='right', va='bottom', fontsize=10, fontweight='bold')
ax.text(xmax - dx, ymax - dy, "Forti & volatili", ha='right', va='top', fontsize=10, fontweight='bold')
ax.text(xmin + dx, ymin + dy, "Deboli & stabili", ha='left', va='bottom', fontsize=10, fontweight='bold')
ax.text(xmin + dx, ymax - dy, "Deboli & volatili", ha='left', va='top', fontsize=10, fontweight='bold')

# --- Legenda A–Z: nome + pallino colorato (colore = quadrante) ---
df_sorted = df.sort_values('playername')

handles = [
    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        markerfacecolor=row['color'],
        markeredgecolor='black',   # bordo come nel plot
        markersize=6,
        label=row['playername']
    )
    for _, row in df_sorted.iterrows()
]

N_COLS = 2  # aumenta a 3 se ti serve più compatta

ax.legend(
    handles=handles,
    title="Giocatori (A–Z)",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True,
    fontsize=9,
    title_fontsize=10,
    ncol=N_COLS,
    handletextpad=0.6,
    columnspacing=1.2
)

plt.tight_layout()
plt.savefig('figures/stabilita_performance_kda.png', dpi=150, bbox_inches='tight')
plt.show()


### Osservazione grafico

Il grafico suddivide i giocatori del LEC 2025 in quattro quadranti sulla base di due dimensioni:

- **Asse orizzontale (KDA medio)**: livello medio di performance stagionale
- **Asse verticale (deviazione standard del KDA)**: variabilità della performance nel corso delle partite

Le linee tratteggiate rappresentano le mediane del campionato e suddividono lo spazio in quattro categorie:

1. **Forti & stabili** — KDA medio sopra la mediana e bassa variabilità
2. **Forti & volatili** — KDA medio elevato ma alta variabilità
3. **Deboli & stabili** — KDA medio sotto la mediana ma performance costante
4. **Deboli & volatili** — KDA medio basso e alta variabilità

L'obiettivo dell'analisi consiste nell'identificare i giocatori più stabili, sia in senso positivo (giocatori affidabili e costantemente performanti) sia in senso negativo (giocatori che mantengono una performance costantemente inferiore alla mediana). Tali due quadranti rappresentano i profili più coerenti nel corso della stagione, mentre i quadranti *volatili* identificano giocatori con maggiore oscillazione nelle prestazioni.

## 3.9 Variabilità delle performance individuali

**Quali giocatori presentano la maggiore varianza di performance — alternando partite molto buone a partite molto negative?**

In [ ]:
# Ordino per varianza decrescente
top10_var = (
    player_stability
    .sort_values('std_kda', ascending=False)
    .head(10)
    .sort_values('std_kda')  # per avere il più alto in cima nel barh
)

plt.figure(figsize=(8,6))
plt.barh(top10_var['playername'], top10_var['std_kda'])

plt.xlabel("Deviazione standard KDA")
plt.ylabel("Player")
plt.title("Top 10 player per maggiore varianza di performance (KDA) - LEC 2025")

plt.tight_layout()
plt.savefig('figures/top10_giocatori_varianza_kda.png', dpi=150, bbox_inches='tight')
plt.show()

### Osservazione grafico

Il grafico mostra i 10 giocatori con la maggiore deviazione standard del KDA nel LEC 2025, ovvero quelli che presentano la più elevata variabilità nelle performance tra una partita e l'altra.

In cima alla classifica si collocano **Supa**, seguito da **SkewMond** e **Ice**, che risultano i player con la maggiore oscillazione tra partite molto positive e partite meno performanti. Un'elevata deviazione standard indica che il rendimento del giocatore non è costante: il player può alternare prestazioni dominanti a cali significativi, può avere un impatto estremamente alto in alcune partite ma non garantire affidabilità nel lungo periodo. Tale profilo può essere sinteticamente descritto come **ad alto impatto ma volatile**.

## 3.10 Giocatori *clutch* nelle partite lunghe

**Esistono giocatori *clutch*, ovvero con performance migliori nelle partite di durata superiore ai 35 minuti?**

In [ ]:
# -----------------------------
# 1) Preparo dataset "clutch"
# -----------------------------
clutch_df = lec_df[['playername', 'position', 'kills', 'deaths', 'assists', 'gamelength']].copy()

# Escludo righe team
clutch_df = clutch_df[clutch_df['position'] != 'team'].copy()

# Converto durata in minuti
clutch_df['gamelength_min'] = clutch_df['gamelength'] / 60

# Definisco fase: long >35 min
clutch_df['game_phase'] = np.where(clutch_df['gamelength_min'] > 35, 'long', 'short')

# KDA per partita (gestione deaths=0)
clutch_df['kda'] = (clutch_df['kills'] + clutch_df['assists']) / clutch_df['deaths'].replace(0, np.nan)

# Tengo solo righe valide
clutch_df = clutch_df.dropna(subset=['playername', 'kda', 'game_phase'])

# ----------------------------------------
# 2) Aggrego KDA medio e numero game per fase
# ----------------------------------------
player_phase = (
    clutch_df
    .groupby(['playername', 'game_phase'], as_index=False)
    .agg(
        games=('kda', 'size'),
        avg_kda=('kda', 'mean')
    )
)

# Pivot medie + pivot conteggi
avg_pivot = player_phase.pivot(index='playername', columns='game_phase', values='avg_kda').reindex(columns=['short', 'long'])
cnt_pivot = player_phase.pivot(index='playername', columns='game_phase', values='games').reindex(columns=['short', 'long'])

# Dataset finale
result_df = (
    avg_pivot.join(cnt_pivot, rsuffix='_games')
    .rename(columns={'short_games': 'games_short', 'long_games': 'games_long'})
    .reset_index()
)

# Delta KDA (Long - Short)
result_df['delta_kda'] = result_df['long'] - result_df['short']

# Clutch score "pesato" per robustezza (penalizza pochi game long)
result_df['clutch_score'] = result_df['delta_kda'] * np.sqrt(result_df['games_long'])

# ----------------------------------------
# 3) Filtro robustezza + Top 10
# ----------------------------------------
MIN_SHORT = 10
MIN_LONG = 10

filtered = result_df[
    (result_df['games_short'] >= MIN_SHORT) &
    (result_df['games_long'] >= MIN_LONG)
].copy()

top10 = (
    filtered
    .sort_values('clutch_score', ascending=False)
    .head(10)
    .sort_values('clutch_score')  # per barh
)

# ----------------------------------------
# 4) Grafico: Top 10 clutch (pesato) + n long
# ----------------------------------------
plt.figure(figsize=(9, 6))
labels = [f"{p} (n={int(n)})" for p, n in zip(top10['playername'], top10['games_long'])]

plt.barh(labels, top10['clutch_score'])
plt.axvline(0, linestyle='--')

plt.xlabel("Clutch Score = (KDA long - KDA short) × √(n long)")
plt.ylabel("Player (partite long)")
plt.title(f"Top 10 player 'clutch' in partite >35 min (min {MIN_SHORT} short, {MIN_LONG} long) - LEC 2025")

plt.tight_layout()
plt.savefig('figures/top10_giocatori_clutch_partite_lunghe.png', dpi=150, bbox_inches='tight')
plt.show()

# (Opzionale) tabella risultati
top10[['playername', 'games_short', 'games_long', 'short', 'long', 'delta_kda', 'clutch_score']].sort_values('clutch_score', ascending=False)


### Osservazione grafico

Il grafico evidenzia i giocatori che migliorano significativamente il proprio KDA nelle partite di durata superiore ai 35 minuti, ponderando il miglioramento per il numero di partite disputate in tale contesto.

**Carlisen** risulta il player più *clutch* del LEC 2025, mostrando un incremento di oltre 3 punti di KDA nelle partite lunghe. Anche **Flakked** e **Sheo** evidenziano un forte miglioramento, suggerendo una maggiore efficacia in situazioni di late game. Particolarmente interessante il caso di **Ice**, che pur partendo da un KDA già elevato nelle partite brevi, migliora ulteriormente nelle partite lunghe, confermandosi un giocatore ad alto impatto nelle fasi avanzate del match.

# 4. Modellazione predittiva

L'ultima sezione del notebook impiega tecniche di *machine learning* per individuare le variabili che meglio spiegano l'esito di una partita LEC 2025. Viene addestrato un classificatore *Random Forest* sia sull'intero insieme di feature di fine partita, sia sul sottoinsieme delle sole variabili rilevabili al minuto 15, al fine di valutare la fattibilità di un modello predittivo basato unicamente sull'early game.

## 4.1 Variabili esplicative dell'esito di una partita

**Quali metriche spiegano meglio la vittoria nel LEC 2025? — analisi di *feature importance* tramite *Random Forest*.**

In [ ]:
team_df = lec_df[lec_df['position'] == 'team'].copy()
team_df.head()

In [ ]:
features = [
    'golddiffat15',
    'kills',
    'deaths',
    'assists',
    'dragons',
    'barons',
    'heralds',
    'towers'
]

X = team_df[features].copy()
y = team_df['result']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)

importances = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances

In [ ]:
plt.figure(figsize=(8,6))
importances.sort_values().plot(kind='barh')

plt.title("Feature Importance - Vittoria LEC 2025")
plt.xlabel("Importanza relativa")
plt.tight_layout()
plt.savefig('figures/feature_importance_modello_completo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

# Predizioni sul test set
y_pred = rf.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy modello Random Forest:", round(accuracy, 3))


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
# =========================
# 1️⃣ Predizioni
# =========================
y_pred = rf.predict(X_test)

# =========================
# 2️⃣ Confusion Matrix (numerica)
# =========================
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix (valori assoluti):")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# =========================
# 3️⃣ Visualizzazione sklearn
# =========================
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="viridis")

plt.title("Confusion Matrix - Random Forest (LEC 2025)")
plt.savefig('figures/confusion_matrix_modello_completo.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.2 Modello predittivo basato sui soli dati al minuto 15

**È possibile costruire un modello predittivo dell'esito di una partita utilizzando esclusivamente i dati disponibili al minuto 15?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

# =========================
# 0) DATASET TEAM-LEVEL (LEC)
# =========================
# Se lec_df è già solo LEC, questo filtro non cambia nulla.
df = lec_df.copy()
if 'league' in df.columns:
    df = df[df['league'] == 'LEC'].copy()

# Solo righe team (una riga per team per game)
df = df[df['position'] == 'team'].copy()

# Target
y = df['result'].astype(int)

# =========================
# 1) FEATURE CANDIDATE @15
# =========================
# Mettiamo una lista "desiderata" tipica dei dataset LoL.
# Il codice userà solo quelle realmente presenti nel tuo df.
candidate_15 = [
    'golddiffat15',
    'csdiffat15',
    'xpdiffat15',
    'killsat15',
    'deathsat15',
    'assistsat15',
    'goldat15',
    'csat15',
    'xpat15',
    'firstblood',
    'firstdragon',
    'firstherald',
    'firsttower',
    'firstmidtower',
    'firsttothreetowers',
    'firstturret',
]

# Prendiamo solo le colonne che esistono davvero
features_15 = [c for c in candidate_15 if c in df.columns]

# Se vuoi essere ancora più "strict", puoi anche filtrare solo colonne che contengono "at15" o "first"
# features_15 = [c for c in df.columns if ('at15' in c.lower()) or c.lower().startswith('first')]

print("Feature @15 trovate e usate:", features_15)

if len(features_15) == 0:
    raise ValueError(
        "Non ho trovato feature @15 nel dataframe. Controlla i nomi colonne (es. golddiffat15, csdiffat15, xpdiffat15...)."
    )

X = df[features_15].copy()

# =========================
# 2) CLEANING MINIMO
# =========================
# Convertiamo eventuali boolean/string a numerico (es. 'Yes'/'No')
for col in X.columns:
    if X[col].dtype == 'object':
        # prova a convertire in numerico, altrimenti mappa yes/no
        X[col] = X[col].replace({'Yes': 1, 'No': 0, 'TRUE': 1, 'FALSE': 0, True: 1, False: 0})
        X[col] = pd.to_numeric(X[col], errors='coerce')

# Fill NaN con mediana (robusto)
X = X.fillna(X.median(numeric_only=True))

# =========================
# 3) TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # mantiene proporzione win/loss
)

# =========================
# 4) MODELLO (Random Forest)
# =========================
rf_15 = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight='balanced'  # utile se win/loss non è 50/50
)

rf_15.fit(X_train, y_train)

# =========================
# 5) PREDIZIONI + METRICHE
# =========================
y_pred = rf_15.predict(X_test)
y_proba = rf_15.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("\n=== Metriche modello @15 (LEC 2025) ===")
print("Accuracy:", round(acc, 3))
print("ROC AUC :", round(auc, 3))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=3))

# =========================
# 6) CONFUSION MATRIX (ASSOLUTA)
# =========================
cm = confusion_matrix(y_test, y_pred)

print("\n=== Confusion Matrix (assoluta) ===")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix – Modello @15 minuti (LEC 2025)")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig('figures/confusion_matrix_modello_15min_assoluta.png', dpi=150, bbox_inches='tight')
plt.show()

# =========================
# 7) CONFUSION MATRIX (NORMALIZZATA per riga)
# =========================
cm_norm = confusion_matrix(y_test, y_pred, normalize="true")

disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm)
disp.plot(cmap="Blues", values_format=".2f")
plt.title("Confusion Matrix normalizzata – Modello @15 minuti (LEC 2025)")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig('figures/confusion_matrix_modello_15min_normalizzata.png', dpi=150, bbox_inches='tight')
plt.show()

# =========================
# 8) FEATURE IMPORTANCE + GRAFICO
# =========================
importances = pd.Series(rf_15.feature_importances_, index=X.columns).sort_values(ascending=False)

print("\n=== Feature importance (@15) ===")
print(importances)

plt.figure(figsize=(8, 6))
importances.sort_values().plot(kind="barh")
plt.title("Feature importance – Modello @15 minuti (LEC 2025)")
plt.xlabel("Importanza relativa")
plt.tight_layout()
plt.savefig('figures/feature_importance_modello_15min.png', dpi=150, bbox_inches='tight')
plt.show()


# Riepilogo dei file prodotti

Il presente notebook produce, oltre alle visualizzazioni *inline*, una serie di file immagine salvati nella sottocartella `figures/` per uso in relazioni e documenti esterni.

## Notebook

- `G2_analisi_clean.ipynb` — notebook pulito e riorganizzato

## Grafici esportati (cartella `figures/`)

- `figures/distribuzione_partite_per_split.png`
- `figures/distribuzione_durata_partite.png`
- `figures/boxplot_durata_partite_per_split.png`
- `figures/winrate_blue_vs_red_per_split.png`
- `figures/winrate_blue_vs_red_per_fascia_durata.png`
- `figures/distribuzione_kills_totali.png`
- `figures/impatto_firstblood_sul_risultato.png`
- `figures/frequenza_partite_snowball.png`
- `figures/impatto_firstbaron_sul_risultato.png`
- `figures/partite_con_baron_o_dragonsoul.png`
- `figures/gold_diff_15min_per_team.png`
- `figures/gold_diff_15min_vittorie_vs_sconfitte.png`
- `figures/conversion_rate_vantaggio_per_team.png`
- `figures/recovery_rate_svantaggio_per_team.png`
- `figures/dragon_control_rate_per_team.png`
- `figures/baron_conversion_rate_per_team.png`
- `figures/winrate_per_fascia_vision_score.png`
- `figures/durata_media_partite_per_team.png`
- `figures/stile_di_gioco_team_kpm_vs_dpm.png`
- `figures/low_kill_vs_objective_efficiency.png`
- `figures/top15_giocatori_gold_diff_15.png`
- `figures/top15_jungler_kill_participation.png`
- `figures/damage_share_medio_per_ruolo.png`
- `figures/top15_support_vspm.png`
- `figures/scatter_kda_vs_winrate.png`
- `figures/scatter_damageshare_vs_winrate.png`
- `figures/stabilita_performance_kda.png`
- `figures/top10_giocatori_varianza_kda.png`
- `figures/top10_giocatori_clutch_partite_lunghe.png`
- `figures/feature_importance_modello_completo.png`
- `figures/confusion_matrix_modello_completo.png`
- `figures/confusion_matrix_modello_15min_assoluta.png`
- `figures/confusion_matrix_modello_15min_normalizzata.png`
- `figures/feature_importance_modello_15min.png`
